# MorphoGraph-MOF (CHGNet), symmetric-slab version, Colab

Same as `MorphoGraph_MOF_colab_4MOFs.ipynb`, with four changes to the science cell, each marked
`SYMMETRIC-SLAB CHANGE` in the code:

1. A slab that is symmetric as cut (same termination on both faces) counts as ONE termination even if
   relaxation breaks its symmetry, so it gives gamma = pair excess / 2 directly. pymatgen's symmetrize
   option is also requested; for MOF-5, HKUST-1 and UiO-66 it returns the same slabs as the plain cut
   (dry run: 2_python/morphology/slab_options_check.py), and the duplicates are skipped, not relaxed.
2. When the pair equations are still underdetermined, the termination energies that they do fix
   (null-space test) are kept instead of discarding the whole facet.
3. A facet class with no fixed energy falls back to half its lowest pair excess. This is an upper
   bound on its lowest termination energy, and the resulting shape is labelled
   `CHGNet_upper_bound_estimate_Wulff_PARTLY_PAIR_AVERAGED`. `metadata.json` records the source of
   every directed energy (`fitted`, `partial_fit_upper_bound`, `pair_average_upper_bound`).
4. Terminations that cut a linker's C-C, C-H, C-O or C-N bonds are skipped when pymatgen can
   find any other (`KEEP_LINKERS_INTACT`). The first runs had loose H, CH, C2H2 and CO2 pieces in
   most slabs because the cutter sliced through linkers.

Slab thickness stays at 15 A (about one to two framework layers); the energies are not
thickness-converged. A second run at 30 A on one MOF would test that.

What to expect, from the earlier runs: the few symmetric slabs are high-energy terminations (1.4 to 9
times the cleavage bound of their face), so most facets will still get the upper-bound estimate.
Keeping linkers whole leaves fewer slabs, so the run should be shorter than before.

Runs HKUST-1 first, then MOF-5 and UiO-66.
Run all cells; each MOF's zip downloads when it finishes.


In [ ]:
# Install, then check the packages and the GPU.
%pip install -q pymatgen chgnet py3Dmol scipy pandas ipython
import pymatgen, chgnet, scipy, pandas, torch
print("CHGNet", getattr(chgnet, "__version__", "?"), "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (CPU, slow)")


In [ ]:
# ---- SETTINGS: edit here, not in the science cell ---------------------------------
MOFS_TO_RUN = ["HKUST-1", "MOF-5", "UiO-66"]   # remove any you want to skip; order = run order

SETTINGS = {
    "MAX_MILLER_INDEX": 1,          # original default 2. 1 = {100},{110},{111}; 2 is far slower
    "SLAB_THICKNESS_A": 15.0,       # original default
    "VACUUM_A": 20.0,               # original default
    "FORCE_TOL_EV_A": 0.05,         # original default
    "MAX_RELAX_STEPS": 500,         # original default
    "PARTICLE_RADIUS_A": 15.0,      # original default (visualisation only)
    "DISPLAY_REPEATS": 2,           # original default
    "SYMPREC_A": 0.1,               # original default
    "SURFACE_FINGERPRINT_DEPTH_A": 4.0,
    "FINGERPRINT_DISTANCE_TOL_A": 0.1,
    "PAIR_FIT_RMS_TOL_EV_A2": 0.02,
    "TERMINATION_SHIFT_TOL_FRACTIONAL": 0.1,
    "SYMMETRIC_SLABS": True,        # added: also build slabs with the same termination on both faces
    "PAIR_AVERAGE_FALLBACK": True,  # added: half the lowest pair excess where no gamma is fixed (upper bound)
    "KEEP_LINKERS_INTACT": True,    # added: never cut C-C, C-H, C-O, C-N bonds (no loose H/CH/CO2 pieces)
}


In [ ]:
# Embedded CIFs (copied unchanged from the repository's 2_python/ folder).
CIFS = {
"ZIF-8": "data_ZIF-8\n\n_audit_creation_method RASPA-1.0\n_audit_creation_date 2011-3-6\n_audit_author_name 'David Dubbeldam'\n\n_citation_author_name        'K.S. Park, Z. Ni, A.P. Cote, J.Y. Choi, R.D. Huang, F.J. Uribe-Romo, H.K. Chae, M. O’Keeffe, and O.M. Yaghi'\n_citation_title              'Exceptional chemical and thermal stability of zeolitic imidazolate frameworks'\n_citation_journal_abbrev     'P. Natl. Acad. Sci. USA'\n_citation_journal_volume     103\n_citation_journal_number     27\n_citation_page_first         10186\n_citation_page_last          10191\n_citation_year               2006\n\n_cell_length_a    16.991\n_cell_length_b    16.991\n_cell_length_c    16.991\n_cell_angle_alpha 90\n_cell_angle_beta  90\n_cell_angle_gamma 90\n_cell_volume      4905.2\n\n_symmetry_cell_setting          cubic\n_symmetry_space_group_name_Hall 'P 1'\n_symmetry_space_group_name_H-M  'P 1'\n_symmetry_Int_Tables_number     1\n\n_symmetry_equiv_pos_as_xyz 'x,y,z'\n\nloop_\n_atom_site_label\n_atom_site_type_symbol\n_atom_site_fract_x\n_atom_site_fract_y\n_atom_site_fract_z\n_atom_site_charge\nC1       C      0.37705    0.0079     0.62295    0\nC2       C      0.36851    0.89914    0.6875     0\nH2       H      0.3777     0.8589     0.7234     0\nC3       C      0.4061     0.0855     0.5939     0\nN1       N      0.40973    0.96832    0.68278    0\nZn1      Zn     0.5        0          0.75       0\nC1       C      0.0079     0.62295    0.37705    0\nC2       C      0.89914    0.63149    0.3125     0\nH2       H      0.8589     0.6223     0.2766     0\nC3       C      0.0855     0.5939     0.4061     0\nN1       N      0.96832    0.59027    0.31722    0\nZn1      Zn     0          0.5        0.25       0\nC1       C      0.62295    0.9921     0.62295    0\nC2       C      0.63149    0.10086    0.6875     0\nH2       H      0.6223     0.1411     0.7234     0\nC3       C      0.5939     0.9145     0.5939     0\nH3A      H      0.5413     0.9202     0.5743     0\nH3B      H      0.5942     0.8772     0.6364     0\nH3C      H      0.6276     0.8961     0.5525     0\nN1       N      0.59027    0.03168    0.68278    0\nC1       C      0.9921     0.37705    0.37705    0\nC2       C      0.10086    0.36851    0.3125     0\nH2       H      0.1411     0.3777     0.2766     0\nC3       C      0.9145     0.4061     0.4061     0\nH3A      H      0.9202     0.4587     0.4257     0\nH3B      H      0.8772     0.4058     0.3636     0\nH3C      H      0.8961     0.3724     0.4475     0\nN1       N      0.03168    0.40973    0.31722    0\nC1       C      0.37705    0.9921     0.37705    0\nC2       C      0.36851    0.10086    0.3125     0\nH2       H      0.3777     0.1411     0.2766     0\nC3       C      0.4061     0.9145     0.4061     0\nN1       N      0.40973    0.03168    0.31722    0\nZn1      Zn     0.5        0          0.25       0\nC1       C      0.9921     0.62295    0.62295    0\nC2       C      0.10086    0.63149    0.6875     0\nH2       H      0.1411     0.6223     0.7234     0\nC3       C      0.9145     0.5939     0.5939     0\nN1       N      0.03168    0.59027    0.68278    0\nZn1      Zn     0          0.5        0.75       0\nC1       C      0.62295    0.0079     0.37705    0\nC2       C      0.63149    0.89914    0.3125     0\nH2       H      0.6223     0.8589     0.2766     0\nC3       C      0.5939     0.0855     0.4061     0\nH3A      H      0.5413     0.0798     0.4257     0\nH3B      H      0.5942     0.1228     0.3636     0\nH3C      H      0.6276     0.1039     0.4475     0\nN1       N      0.59027    0.96832    0.31722    0\nC1       C      0.0079     0.37705    0.62295    0\nC2       C      0.89914    0.36851    0.6875     0\nH2       H      0.8589     0.3777     0.7234     0\nC3       C      0.0855     0.4061     0.5939     0\nH3A      H      0.0798     0.4587     0.5743     0\nH3B      H      0.1228     0.4058     0.6364     0\nH3C      H      0.1039     0.3724     0.5525     0\nN1       N      0.96832    0.40973    0.68278    0\nC1       C      0.62295    0.37705    0.0079     0\nC2       C      0.6875     0.36851    0.89914    0\nH2       H      0.7234     0.3777     0.8589     0\nC3       C      0.5939     0.4061     0.0855     0\nN1       N      0.68278    0.40973    0.96832    0\nZn1      Zn     0.75       0.5        0          0\nC1       C      0.37705    0.37705    0.9921     0\nC2       C      0.36851    0.3125     0.10086    0\nH2       H      0.3777     0.2766     0.1411     0\nC3       C      0.4061     0.4061     0.9145     0\nH3A      H      0.4587     0.4257     0.9202     0\nH3B      H      0.4058     0.3636     0.8772     0\nH3C      H      0.3724     0.4475     0.8961     0\nN1       N      0.40973    0.31722    0.03168    0\nZn1      Zn     0.5        0.25       0          0\nC1       C      0.37705    0.62295    0.0079     0\nC2       C      0.3125     0.63149    0.89914    0\nH2       H      0.2766     0.6223     0.8589     0\nC3       C      0.4061     0.5939     0.0855     0\nN1       N      0.31722    0.59027    0.96832    0\nZn1      Zn     0.25       0.5        0          0\nC1       C      0.62295    0.62295    0.9921     0\nC2       C      0.63149    0.6875     0.10086    0\nH2       H      0.6223     0.7234     0.1411     0\nC3       C      0.5939     0.5939     0.9145     0\nH3A      H      0.5413     0.5743     0.9202     0\nH3B      H      0.5942     0.6364     0.8772     0\nH3C      H      0.6276     0.5525     0.8961     0\nN1       N      0.59027    0.68278    0.03168    0\nZn1      Zn     0.5        0.75       0          0\nC2       C      0.6875     0.63149    0.10086    0\nH2       H      0.7234     0.6223     0.1411     0\nN1       N      0.68278    0.59027    0.03168    0\nC2       C      0.63149    0.3125     0.89914    0\nH2       H      0.6223     0.2766     0.8589     0\nH3A      H      0.5413     0.4257     0.0798     0\nH3B      H      0.5942     0.3636     0.1228     0\nH3C      H      0.6276     0.4475     0.1039     0\nN1       N      0.59027    0.31722    0.96832    0\nC2       C      0.3125     0.36851    0.10086    0\nH2       H      0.2766     0.3777     0.1411     0\nN1       N      0.31722    0.40973    0.03168    0\nC2       C      0.36851    0.6875     0.89914    0\nH2       H      0.3777     0.7234     0.8589     0\nH3A      H      0.4587     0.5743     0.0798     0\nH3B      H      0.4058     0.6364     0.1228     0\nH3C      H      0.3724     0.5525     0.1039     0\nN1       N      0.40973    0.68278    0.96832    0\nC2       C      0.89914    0.6875     0.36851    0\nH2       H      0.8589     0.7234     0.3777     0\nH3A      H      0.0798     0.5743     0.4587     0\nH3B      H      0.1228     0.6364     0.4058     0\nH3C      H      0.1039     0.5525     0.3724     0\nN1       N      0.96832    0.68278    0.40973    0\nZn1      Zn     0          0.75       0.5        0\nC2       C      0.89914    0.3125     0.63149    0\nH2       H      0.8589     0.2766     0.6223     0\nN1       N      0.96832    0.31722    0.59027    0\nZn1      Zn     0          0.25       0.5        0\nC2       C      0.3125     0.10086    0.36851    0\nH2       H      0.2766     0.1411     0.3777     0\nH3A      H      0.4257     0.9202     0.4587     0\nH3B      H      0.3636     0.8772     0.4058     0\nH3C      H      0.4475     0.8961     0.3724     0\nN1       N      0.31722    0.03168    0.40973    0\nZn1      Zn     0.25       0          0.5        0\nC2       C      0.10086    0.6875     0.63149    0\nH2       H      0.1411     0.7234     0.6223     0\nH3A      H      0.9202     0.5743     0.5413     0\nH3B      H      0.8772     0.6364     0.5942     0\nH3C      H      0.8961     0.5525     0.6276     0\nN1       N      0.03168    0.68278    0.59027    0\nC2       C      0.6875     0.89914    0.36851    0\nH2       H      0.7234     0.8589     0.3777     0\nN1       N      0.68278    0.96832    0.40973    0\nZn1      Zn     0.75       0          0.5        0\nC2       C      0.10086    0.3125     0.36851    0\nH2       H      0.1411     0.2766     0.3777     0\nN1       N      0.03168    0.31722    0.40973    0\nC2       C      0.3125     0.89914    0.63149    0\nH2       H      0.2766     0.8589     0.6223     0\nH3A      H      0.4257     0.0798     0.5413     0\nH3B      H      0.3636     0.1228     0.5942     0\nH3C      H      0.4475     0.1039     0.6276     0\nN1       N      0.31722    0.96832    0.59027    0\nC2       C      0.6875     0.10086    0.63149    0\nH2       H      0.7234     0.1411     0.6223     0\nN1       N      0.68278    0.03168    0.59027    0\nC1       C      0.87705    0.5079     0.12295    0\nC2       C      0.86851    0.39914    0.1875     0\nH2       H      0.8777     0.3589     0.2234     0\nC3       C      0.9061     0.5855     0.0939     0\nH3A      H      0.9587     0.5798     0.0743     0\nH3B      H      0.9058     0.6228     0.1364     0\nH3C      H      0.8724     0.6039     0.0525     0\nN1       N      0.90973    0.46832    0.18278    0\nC1       C      0.5079     0.12295    0.87705    0\nC2       C      0.39914    0.13149    0.8125     0\nH2       H      0.3589     0.1223     0.7766     0\nC3       C      0.5855     0.0939     0.9061     0\nN1       N      0.46832    0.09027    0.81722    0\nC1       C      0.12295    0.4921     0.12295    0\nC2       C      0.13149    0.60086    0.1875     0\nH2       H      0.1223     0.6411     0.2234     0\nC3       C      0.0939     0.4145     0.0939     0\nH3A      H      0.0413     0.4202     0.0743     0\nH3B      H      0.0942     0.3772     0.1364     0\nH3C      H      0.1276     0.3961     0.0525     0\nN1       N      0.09027    0.53168    0.18278    0\nC1       C      0.4921     0.87705    0.87705    0\nC2       C      0.60086    0.86851    0.8125     0\nH2       H      0.6411     0.8777     0.7766     0\nC3       C      0.4145     0.9061     0.9061     0\nH3A      H      0.4202     0.9587     0.9257     0\nH3B      H      0.3772     0.9058     0.8636     0\nH3C      H      0.3961     0.8724     0.9475     0\nN1       N      0.53168    0.90973    0.81722    0\nC1       C      0.87705    0.4921     0.87705    0\nC2       C      0.86851    0.60086    0.8125     0\nH2       H      0.8777     0.6411     0.7766     0\nC3       C      0.9061     0.4145     0.9061     0\nH3A      H      0.9587     0.4202     0.9257     0\nH3B      H      0.9058     0.3772     0.8636     0\nH3C      H      0.8724     0.3961     0.9475     0\nN1       N      0.90973    0.53168    0.81722    0\nC1       C      0.4921     0.12295    0.12295    0\nC2       C      0.60086    0.13149    0.1875     0\nH2       H      0.6411     0.1223     0.2234     0\nC3       C      0.4145     0.0939     0.0939     0\nN1       N      0.53168    0.09027    0.18278    0\nC1       C      0.12295    0.5079     0.87705    0\nC2       C      0.13149    0.39914    0.8125     0\nH2       H      0.1223     0.3589     0.7766     0\nC3       C      0.0939     0.5855     0.9061     0\nN1       N      0.09027    0.46832    0.81722    0\nC1       C      0.5079     0.87705    0.12295    0\nC2       C      0.39914    0.86851    0.1875     0\nH2       H      0.3589     0.8777     0.2234     0\nC3       C      0.5855     0.9061     0.0939     0\nN1       N      0.46832    0.90973    0.18278    0\nC1       C      0.12295    0.87705    0.5079     0\nC2       C      0.1875     0.86851    0.39914    0\nH2       H      0.2234     0.8777     0.3589     0\nC3       C      0.0939     0.9061     0.5855     0\nN1       N      0.18278    0.90973    0.46832    0\nC1       C      0.87705    0.87705    0.4921     0\nC2       C      0.86851    0.8125     0.60086    0\nH2       H      0.8777     0.7766     0.6411     0\nC3       C      0.9061     0.9061     0.4145     0\nH3A      H      0.9587     0.9257     0.4202     0\nH3B      H      0.9058     0.8636     0.3772     0\nH3C      H      0.8724     0.9475     0.3961     0\nN1       N      0.90973    0.81722    0.53168    0\nC1       C      0.87705    0.12295    0.5079     0\nC2       C      0.8125     0.13149    0.39914    0\nH2       H      0.7766     0.1223     0.3589     0\nC3       C      0.9061     0.0939     0.5855     0\nN1       N      0.81722    0.09027    0.46832    0\nC1       C      0.12295    0.12295    0.4921     0\nC2       C      0.13149    0.1875     0.60086    0\nH2       H      0.1223     0.2234     0.6411     0\nC3       C      0.0939     0.0939     0.4145     0\nH3A      H      0.0413     0.0743     0.4202     0\nH3B      H      0.0942     0.1364     0.3772     0\nH3C      H      0.1276     0.0525     0.3961     0\nN1       N      0.09027    0.18278    0.53168    0\nC2       C      0.1875     0.13149    0.60086    0\nH2       H      0.2234     0.1223     0.6411     0\nN1       N      0.18278    0.09027    0.53168    0\nC2       C      0.13149    0.8125     0.39914    0\nH2       H      0.1223     0.7766     0.3589     0\nH3A      H      0.0413     0.9257     0.5798     0\nH3B      H      0.0942     0.8636     0.6228     0\nH3C      H      0.1276     0.9475     0.6039     0\nN1       N      0.09027    0.81722    0.46832    0\nC2       C      0.8125     0.86851    0.60086    0\nH2       H      0.7766     0.8777     0.6411     0\nN1       N      0.81722    0.90973    0.53168    0\nC2       C      0.86851    0.1875     0.39914    0\nH2       H      0.8777     0.2234     0.3589     0\nH3A      H      0.9587     0.0743     0.5798     0\nH3B      H      0.9058     0.1364     0.6228     0\nH3C      H      0.8724     0.0525     0.6039     0\nN1       N      0.90973    0.18278    0.46832    0\nC2       C      0.39914    0.1875     0.86851    0\nH2       H      0.3589     0.2234     0.8777     0\nH3A      H      0.5798     0.0743     0.9587     0\nH3B      H      0.6228     0.1364     0.9058     0\nH3C      H      0.6039     0.0525     0.8724     0\nN1       N      0.46832    0.18278    0.90973    0\nC2       C      0.39914    0.8125     0.13149    0\nH2       H      0.3589     0.7766     0.1223     0\nH3A      H      0.5798     0.9257     0.0413     0\nH3B      H      0.6228     0.8636     0.0942     0\nH3C      H      0.6039     0.9475     0.1276     0\nN1       N      0.46832    0.81722    0.09027    0\nC2       C      0.8125     0.60086    0.86851    0\nH2       H      0.7766     0.6411     0.8777     0\nN1       N      0.81722    0.53168    0.90973    0\nC2       C      0.60086    0.1875     0.13149    0\nH2       H      0.6411     0.2234     0.1223     0\nH3A      H      0.4202     0.0743     0.0413     0\nH3B      H      0.3772     0.1364     0.0942     0\nH3C      H      0.3961     0.0525     0.1276     0\nN1       N      0.53168    0.18278    0.09027    0\nC2       C      0.1875     0.39914    0.86851    0\nH2       H      0.2234     0.3589     0.8777     0\nH3A      H      0.0743     0.5798     0.9587     0\nH3B      H      0.1364     0.6228     0.9058     0\nH3C      H      0.0525     0.6039     0.8724     0\nN1       N      0.18278    0.46832    0.90973    0\nC2       C      0.60086    0.8125     0.86851    0\nH2       H      0.6411     0.7766     0.8777     0\nN1       N      0.53168    0.81722    0.90973    0\nC2       C      0.8125     0.39914    0.13149    0\nH2       H      0.7766     0.3589     0.1223     0\nN1       N      0.81722    0.46832    0.09027    0\nC2       C      0.1875     0.60086    0.13149    0\nH2       H      0.2234     0.6411     0.1223     0\nN1       N      0.18278    0.53168    0.09027    0\n",
"HKUST-1": "#======================================================================\n\n# CRYSTAL DATA\n\n#----------------------------------------------------------------------\n\ndata_VESTA_phase_1\n\n\n_pd_phase_name                         'HKUST-1 PBEsol                        '\n_cell_length_a                         18.59484\n_cell_length_b                         18.59490\n_cell_length_c                         18.59496\n_cell_angle_alpha                      59.99992\n_cell_angle_beta                       60.00001\n_cell_angle_gamma                      60.00034\n_symmetry_space_group_name_H-M         'P 1'\n_symmetry_Int_Tables_number            1\n\nloop_\n_symmetry_equiv_pos_as_xyz\n   'x, y, z'\n\nloop_\n   _atom_site_label\n   _atom_site_occupancy\n   _atom_site_fract_x\n   _atom_site_fract_y\n   _atom_site_fract_z\n   _atom_site_adp_type\n   _atom_site_B_iso_or_equiv\n   _atom_site_type_symbol\n   O1         1.0     0.874414      0.020501      0.613727     Biso  1.000000 O\n   O2         1.0     0.020509      0.874415      0.491353     Biso  1.000000 O\n   O3         1.0     0.613724      0.491353      0.874415     Biso  1.000000 O\n   O4         1.0     0.491359      0.613727      0.020501     Biso  1.000000 O\n   O5         1.0     0.613727      0.874414      0.020504     Biso  1.000000 O\n   O6         1.0     0.491355      0.020504      0.874414     Biso  1.000000 O\n   O7         1.0     0.874412      0.613725      0.491356     Biso  1.000000 O\n   O8         1.0     0.020508      0.491356      0.613725     Biso  1.000000 O\n   O9         1.0     0.020501      0.613726      0.874416     Biso  1.000000 O\n   O10        1.0     0.874415      0.491355      0.020504     Biso  1.000000 O\n   O11        1.0     0.491356      0.874416      0.613726     Biso  1.000000 O\n   O12        1.0     0.613726      0.020504      0.491355     Biso  1.000000 O\n   O13        1.0     0.125585      0.979492      0.508647     Biso  1.000000 O\n   O14        1.0     0.979498      0.125586      0.386274     Biso  1.000000 O\n   O15        1.0     0.386276      0.508647      0.979492     Biso  1.000000 O\n   O16        1.0     0.508641      0.386274      0.125586     Biso  1.000000 O\n   O17        1.0     0.386275      0.125586      0.508646     Biso  1.000000 O\n   O18        1.0     0.508642      0.979496      0.386274     Biso  1.000000 O\n   O19        1.0     0.125588      0.386274      0.979496     Biso  1.000000 O\n   O20        1.0     0.979492      0.508646      0.125586     Biso  1.000000 O\n   O21        1.0     0.979498      0.386273      0.508646     Biso  1.000000 O\n   O22        1.0     0.125583      0.508646      0.386273     Biso  1.000000 O\n   O23        1.0     0.508645      0.125586      0.979496     Biso  1.000000 O\n   O24        1.0     0.386273      0.979496      0.125586     Biso  1.000000 O\n   O25        1.0     0.125586      0.979499      0.386273     Biso  1.000000 O\n   O26        1.0     0.979491      0.125585      0.508647     Biso  1.000000 O\n   O27        1.0     0.386276      0.508647      0.125585     Biso  1.000000 O\n   O28        1.0     0.508641      0.386273      0.979499     Biso  1.000000 O\n   O29        1.0     0.386273      0.125586      0.979496     Biso  1.000000 O\n   O30        1.0     0.508645      0.979496      0.125586     Biso  1.000000 O\n   O31        1.0     0.125588      0.386275      0.508644     Biso  1.000000 O\n   O32        1.0     0.979492      0.508644      0.386275     Biso  1.000000 O\n   O33        1.0     0.979499      0.386274      0.125584     Biso  1.000000 O\n   O34        1.0     0.125585      0.508645      0.979496     Biso  1.000000 O\n   O35        1.0     0.508644      0.125584      0.386274     Biso  1.000000 O\n   O36        1.0     0.386274      0.979496      0.508645     Biso  1.000000 O\n   O37        1.0     0.874415      0.020508      0.491353     Biso  1.000000 O\n   O38        1.0     0.020502      0.874414      0.613726     Biso  1.000000 O\n   O39        1.0     0.613724      0.491353      0.020508     Biso  1.000000 O\n   O40        1.0     0.491359      0.613726      0.874414     Biso  1.000000 O\n   O41        1.0     0.613725      0.874414      0.491354     Biso  1.000000 O\n   O42        1.0     0.491358      0.020504      0.613726     Biso  1.000000 O\n   O43        1.0     0.874412      0.613726      0.020504     Biso  1.000000 O\n   O44        1.0     0.020507      0.491354      0.874414     Biso  1.000000 O\n   O45        1.0     0.020502      0.613727      0.491354     Biso  1.000000 O\n   O46        1.0     0.874417      0.491354      0.613727     Biso  1.000000 O\n   O47        1.0     0.491355      0.874414      0.020504     Biso  1.000000 O\n   O48        1.0     0.613727      0.020504      0.874414     Biso  1.000000 O\n   C1         1.0     0.700152      0.029701      0.700153     Biso  1.000000 C\n   C2         1.0     0.029702      0.700153      0.569994     Biso  1.000000 C\n   C3         1.0     0.700152      0.569994      0.700153     Biso  1.000000 C\n   C4         1.0     0.569995      0.700153      0.029701     Biso  1.000000 C\n   C5         1.0     0.700152      0.700153      0.029701     Biso  1.000000 C\n   C6         1.0     0.569996      0.029701      0.700153     Biso  1.000000 C\n   C7         1.0     0.700151      0.700152      0.569994     Biso  1.000000 C\n   C8         1.0     0.029704      0.569994      0.700152     Biso  1.000000 C\n   C9         1.0     0.029702      0.700151      0.700152     Biso  1.000000 C\n   C10        1.0     0.700153      0.569994      0.029701     Biso  1.000000 C\n   C11        1.0     0.569995      0.700152      0.700151     Biso  1.000000 C\n   C12        1.0     0.700153      0.029701      0.569994     Biso  1.000000 C\n   C13        1.0     0.299847      0.970299      0.430006     Biso  1.000000 C\n   C14        1.0     0.970298      0.299849      0.299848     Biso  1.000000 C\n   C15        1.0     0.299847      0.430006      0.970299     Biso  1.000000 C\n   C16        1.0     0.430005      0.299848      0.299849     Biso  1.000000 C\n   C17        1.0     0.299849      0.299848      0.430006     Biso  1.000000 C\n   C18        1.0     0.430004      0.970299      0.299847     Biso  1.000000 C\n   C19        1.0     0.299848      0.299847      0.970299     Biso  1.000000 C\n   C20        1.0     0.970296      0.430006      0.299848     Biso  1.000000 C\n   C21        1.0     0.970298      0.299847      0.430006     Biso  1.000000 C\n   C22        1.0     0.299848      0.430006      0.299847     Biso  1.000000 C\n   C23        1.0     0.430005      0.299847      0.970299     Biso  1.000000 C\n   C24        1.0     0.299848      0.970299      0.299847     Biso  1.000000 C\n   C25        1.0     0.837747      0.023831      0.569209     Biso  1.000000 C\n   C26        1.0     0.023831      0.837747      0.569208     Biso  1.000000 C\n   C27        1.0     0.569212      0.569208      0.837747     Biso  1.000000 C\n   C28        1.0     0.569212      0.569209      0.023831     Biso  1.000000 C\n   C29        1.0     0.569210      0.837747      0.023832     Biso  1.000000 C\n   C30        1.0     0.569210      0.023832      0.837747     Biso  1.000000 C\n   C31        1.0     0.837746      0.569209      0.569212     Biso  1.000000 C\n   C32        1.0     0.023832      0.569212      0.569209     Biso  1.000000 C\n   C33        1.0     0.023832      0.569210      0.837747     Biso  1.000000 C\n   C34        1.0     0.837746      0.569209      0.023831     Biso  1.000000 C\n   C35        1.0     0.569210      0.837747      0.569210     Biso  1.000000 C\n   C36        1.0     0.569212      0.023831      0.569209     Biso  1.000000 C\n   C37        1.0     0.162253      0.976169      0.430791     Biso  1.000000 C\n   C38        1.0     0.976169      0.162253      0.430792     Biso  1.000000 C\n   C39        1.0     0.430788      0.430791      0.976169     Biso  1.000000 C\n   C40        1.0     0.430788      0.430792      0.162253     Biso  1.000000 C\n   C41        1.0     0.430790      0.162253      0.430790     Biso  1.000000 C\n   C42        1.0     0.430788      0.976169      0.430791     Biso  1.000000 C\n   C43        1.0     0.162254      0.430791      0.976169     Biso  1.000000 C\n   C44        1.0     0.976168      0.430790      0.162253     Biso  1.000000 C\n   C45        1.0     0.976168      0.430788      0.430791     Biso  1.000000 C\n   C46        1.0     0.162254      0.430791      0.430788     Biso  1.000000 C\n   C47        1.0     0.430790      0.162253      0.976168     Biso  1.000000 C\n   C48        1.0     0.430790      0.976168      0.162253     Biso  1.000000 C\n   C49        1.0     0.743404      0.029630      0.613482     Biso  1.000000 C\n   C50        1.0     0.029631      0.743404      0.613481     Biso  1.000000 C\n   C51        1.0     0.613483      0.613481      0.743404     Biso  1.000000 C\n   C52        1.0     0.613482      0.613482      0.029630     Biso  1.000000 C\n   C53        1.0     0.613482      0.743403      0.029631     Biso  1.000000 C\n   C54        1.0     0.613482      0.029631      0.743403     Biso  1.000000 C\n   C55        1.0     0.743404      0.613481      0.613482     Biso  1.000000 C\n   C56        1.0     0.029632      0.613482      0.613481     Biso  1.000000 C\n   C57        1.0     0.029632      0.613482      0.743403     Biso  1.000000 C\n   C58        1.0     0.743404      0.613482      0.029630     Biso  1.000000 C\n   C59        1.0     0.613482      0.743403      0.613482     Biso  1.000000 C\n   C60        1.0     0.613483      0.029630      0.613482     Biso  1.000000 C\n   C61        1.0     0.256595      0.970370      0.386518     Biso  1.000000 C\n   C62        1.0     0.970369      0.256595      0.386519     Biso  1.000000 C\n   C63        1.0     0.386518      0.386518      0.970370     Biso  1.000000 C\n   C64        1.0     0.386517      0.386519      0.256595     Biso  1.000000 C\n   C65        1.0     0.386518      0.256596      0.386518     Biso  1.000000 C\n   C66        1.0     0.386517      0.970370      0.386518     Biso  1.000000 C\n   C67        1.0     0.256595      0.386518      0.970370     Biso  1.000000 C\n   C68        1.0     0.970368      0.386518      0.256596     Biso  1.000000 C\n   C69        1.0     0.970368      0.386518      0.386519     Biso  1.000000 C\n   C70        1.0     0.256595      0.386519      0.386518     Biso  1.000000 C\n   C71        1.0     0.386518      0.256595      0.970369     Biso  1.000000 C\n   C72        1.0     0.386518      0.970369      0.256595     Biso  1.000000 C\n   H1         1.0     0.734615      0.028085      0.734614     Biso  1.000000 H\n   H2         1.0     0.028085      0.734617      0.502682     Biso  1.000000 H\n   H3         1.0     0.734616      0.502682      0.734617     Biso  1.000000 H\n   H4         1.0     0.502686      0.734614      0.028085     Biso  1.000000 H\n   H5         1.0     0.734615      0.734615      0.028085     Biso  1.000000 H\n   H6         1.0     0.502685      0.028085      0.734615     Biso  1.000000 H\n   H7         1.0     0.734615      0.734615      0.502685     Biso  1.000000 H\n   H8         1.0     0.028086      0.502685      0.734615     Biso  1.000000 H\n   H9         1.0     0.028085      0.734615      0.734615     Biso  1.000000 H\n   H10        1.0     0.734616      0.502682      0.028085     Biso  1.000000 H\n   H11        1.0     0.502686      0.734615      0.734615     Biso  1.000000 H\n   H12        1.0     0.734617      0.028085      0.502682     Biso  1.000000 H\n   H13        1.0     0.265383      0.971915      0.497318     Biso  1.000000 H\n   H14        1.0     0.971915      0.265385      0.265385     Biso  1.000000 H\n   H15        1.0     0.265384      0.497318      0.971915     Biso  1.000000 H\n   H16        1.0     0.497314      0.265385      0.265385     Biso  1.000000 H\n   H17        1.0     0.265385      0.265385      0.497315     Biso  1.000000 H\n   H18        1.0     0.497315      0.971915      0.265385     Biso  1.000000 H\n   H19        1.0     0.265385      0.265385      0.971915     Biso  1.000000 H\n   H20        1.0     0.971914      0.497315      0.265385     Biso  1.000000 H\n   H21        1.0     0.971915      0.265383      0.497318     Biso  1.000000 H\n   H22        1.0     0.265384      0.497318      0.265383     Biso  1.000000 H\n   H23        1.0     0.497314      0.265385      0.971915     Biso  1.000000 H\n   H24        1.0     0.265385      0.971915      0.265385     Biso  1.000000 H\n   Cu1        1.0     0.999998      0.999998      0.565198     Biso  1.000000 Cu\n   Cu2        1.0     0.565194      0.434802      0.000002     Biso  1.000000 Cu\n   Cu3        1.0     0.565196      0.000000      1.000000     Biso  1.000000 Cu\n   Cu4        1.0     0.999995      0.565195      0.434805     Biso  1.000000 Cu\n   Cu5        1.0     0.999997      0.565197      0.000001     Biso  1.000000 Cu\n   Cu6        1.0     0.565195      0.999999      0.434803     Biso  1.000000 Cu\n   Cu7        1.0     0.000002      0.000002      0.434802     Biso  1.000000 Cu\n   Cu8        1.0     0.434806      0.565198      0.999998     Biso  1.000000 Cu\n   Cu9        1.0     0.434804      1.000000      0.000000     Biso  1.000000 Cu\n   Cu10       1.0     0.000005      0.434805      0.565195     Biso  1.000000 Cu\n   Cu11       1.0     0.000003      0.434803      0.999999     Biso  1.000000 Cu\n   Cu12       1.0     0.434805      0.000001      0.565197     Biso  1.000000 Cu\n",
"UiO-66": "#======================================================================\n\n# CRYSTAL DATA\n\n#----------------------------------------------------------------------\n\ndata_VESTA_phase_1\n\n\n_pd_phase_name                         'UiO-66 - PBEsol                       '\n_cell_length_a                         14.50863\n_cell_length_b                         14.50863\n_cell_length_c                         14.50863\n_cell_angle_alpha                      60.00000\n_cell_angle_beta                       60.00000\n_cell_angle_gamma                      60.00000\n_symmetry_space_group_name_H-M         'P 1'\n_symmetry_Int_Tables_number            1\n\nloop_\n_symmetry_equiv_pos_as_xyz\n   'x, y, z'\n\nloop_\n   _atom_site_label\n   _atom_site_occupancy\n   _atom_site_fract_x\n   _atom_site_fract_y\n   _atom_site_fract_z\n   _atom_site_adp_type\n   _atom_site_B_iso_or_equiv\n   _atom_site_type_symbol\n   H1         1.0     0.155217      0.580411      0.404711     Biso  1.000000 H\n   H2         1.0     0.580411      0.155217      0.859660     Biso  1.000000 H\n   H3         1.0     0.404711      0.859660      0.155217     Biso  1.000000 H\n   H4         1.0     0.859660      0.404711      0.580411     Biso  1.000000 H\n   H5         1.0     0.404711      0.155217      0.580411     Biso  1.000000 H\n   H6         1.0     0.859660      0.580411      0.155217     Biso  1.000000 H\n   H7         1.0     0.155217      0.404711      0.859660     Biso  1.000000 H\n   H8         1.0     0.580411      0.859660      0.404711     Biso  1.000000 H\n   H9         1.0     0.580411      0.404711      0.155217     Biso  1.000000 H\n   H10        1.0     0.155217      0.859660      0.580411     Biso  1.000000 H\n   H11        1.0     0.859660      0.155217      0.404711     Biso  1.000000 H\n   H12        1.0     0.404711      0.580411      0.859660     Biso  1.000000 H\n   H13        1.0     0.580411      0.155217      0.404711     Biso  1.000000 H\n   H14        1.0     0.155217      0.580411      0.859660     Biso  1.000000 H\n   H15        1.0     0.859660      0.404711      0.155217     Biso  1.000000 H\n   H16        1.0     0.404711      0.859660      0.580411     Biso  1.000000 H\n   H17        1.0     0.155217      0.404711      0.580411     Biso  1.000000 H\n   H18        1.0     0.580411      0.859660      0.155217     Biso  1.000000 H\n   H19        1.0     0.404711      0.155217      0.859660     Biso  1.000000 H\n   H20        1.0     0.859660      0.580411      0.404711     Biso  1.000000 H\n   H21        1.0     0.404711      0.580411      0.155217     Biso  1.000000 H\n   H22        1.0     0.859660      0.155217      0.580411     Biso  1.000000 H\n   H23        1.0     0.155217      0.859660      0.404711     Biso  1.000000 H\n   H24        1.0     0.580411      0.404711      0.859660     Biso  1.000000 H\n   H25        1.0     0.904736      0.904736      0.285792     Biso  1.000000 H\n   H26        1.0     0.904736      0.904736      0.904736     Biso  1.000000 H\n   H27        1.0     0.285792      0.904736      0.904736     Biso  1.000000 H\n   H28        1.0     0.904736      0.285792      0.904736     Biso  1.000000 H\n   C1         1.0     0.091988      0.539726      0.444553     Biso  1.000000 C\n   C2         1.0     0.539726      0.091988      0.923733     Biso  1.000000 C\n   C3         1.0     0.444553      0.923733      0.091988     Biso  1.000000 C\n   C4         1.0     0.923733      0.444553      0.539726     Biso  1.000000 C\n   C5         1.0     0.444553      0.091988      0.539726     Biso  1.000000 C\n   C6         1.0     0.923733      0.539726      0.091988     Biso  1.000000 C\n   C7         1.0     0.091988      0.444553      0.923733     Biso  1.000000 C\n   C8         1.0     0.539726      0.923733      0.444553     Biso  1.000000 C\n   C9         1.0     0.539726      0.444553      0.091988     Biso  1.000000 C\n   C10        1.0     0.091988      0.923733      0.539726     Biso  1.000000 C\n   C11        1.0     0.923733      0.091988      0.444553     Biso  1.000000 C\n   C12        1.0     0.444553      0.539726      0.923733     Biso  1.000000 C\n   C13        1.0     0.539726      0.091988      0.444553     Biso  1.000000 C\n   C14        1.0     0.091988      0.539726      0.923733     Biso  1.000000 C\n   C15        1.0     0.923733      0.444553      0.091988     Biso  1.000000 C\n   C16        1.0     0.444553      0.923733      0.539726     Biso  1.000000 C\n   C17        1.0     0.091988      0.444553      0.539726     Biso  1.000000 C\n   C18        1.0     0.539726      0.923733      0.091988     Biso  1.000000 C\n   C19        1.0     0.444553      0.091988      0.923733     Biso  1.000000 C\n   C20        1.0     0.923733      0.539726      0.444553     Biso  1.000000 C\n   C21        1.0     0.444553      0.539726      0.091988     Biso  1.000000 C\n   C22        1.0     0.923733      0.091988      0.539726     Biso  1.000000 C\n   C23        1.0     0.091988      0.923733      0.444553     Biso  1.000000 C\n   C24        1.0     0.539726      0.444553      0.923733     Biso  1.000000 C\n   C25        1.0     0.299965      0.004529      0.004529     Biso  1.000000 C\n   C26        1.0     0.004529      0.299965      0.690977     Biso  1.000000 C\n   C27        1.0     0.004529      0.690977      0.299965     Biso  1.000000 C\n   C28        1.0     0.690977      0.004529      0.004529     Biso  1.000000 C\n   C29        1.0     0.004529      0.299965      0.004529     Biso  1.000000 C\n   C30        1.0     0.690977      0.004529      0.299965     Biso  1.000000 C\n   C31        1.0     0.299965      0.004529      0.690977     Biso  1.000000 C\n   C32        1.0     0.004529      0.690977      0.004529     Biso  1.000000 C\n   C33        1.0     0.004529      0.004529      0.299965     Biso  1.000000 C\n   C34        1.0     0.299965      0.690977      0.004529     Biso  1.000000 C\n   C35        1.0     0.690977      0.299965      0.004529     Biso  1.000000 C\n   C36        1.0     0.004529      0.004529      0.690977     Biso  1.000000 C\n   C37        1.0     0.007411      0.586820      0.398358     Biso  1.000000 C\n   C38        1.0     0.586820      0.007411      0.007411     Biso  1.000000 C\n   C39        1.0     0.398358      0.007411      0.007411     Biso  1.000000 C\n   C40        1.0     0.007411      0.398358      0.586820     Biso  1.000000 C\n   C41        1.0     0.398358      0.007411      0.586820     Biso  1.000000 C\n   C42        1.0     0.007411      0.586820      0.007411     Biso  1.000000 C\n   C43        1.0     0.007411      0.398358      0.007411     Biso  1.000000 C\n   C44        1.0     0.586820      0.007411      0.398358     Biso  1.000000 C\n   C45        1.0     0.586820      0.398358      0.007411     Biso  1.000000 C\n   C46        1.0     0.007411      0.007411      0.586820     Biso  1.000000 C\n   C47        1.0     0.007411      0.007411      0.398358     Biso  1.000000 C\n   C48        1.0     0.398358      0.586820      0.007411     Biso  1.000000 C\n   O1         1.0     0.081082      0.731998      0.262517     Biso  1.000000 O\n   O2         1.0     0.731998      0.081082      0.924403     Biso  1.000000 O\n   O3         1.0     0.262517      0.924403      0.081082     Biso  1.000000 O\n   O4         1.0     0.924403      0.262517      0.731998     Biso  1.000000 O\n   O5         1.0     0.262517      0.081082      0.731998     Biso  1.000000 O\n   O6         1.0     0.924403      0.731998      0.081082     Biso  1.000000 O\n   O7         1.0     0.081082      0.262517      0.924403     Biso  1.000000 O\n   O8         1.0     0.731998      0.924403      0.262517     Biso  1.000000 O\n   O9         1.0     0.731998      0.262517      0.081082     Biso  1.000000 O\n   O10        1.0     0.081082      0.924403      0.731998     Biso  1.000000 O\n   O11        1.0     0.924403      0.081082      0.262517     Biso  1.000000 O\n   O12        1.0     0.262517      0.731998      0.924403     Biso  1.000000 O\n   O13        1.0     0.731998      0.081082      0.262517     Biso  1.000000 O\n   O14        1.0     0.081082      0.731998      0.924403     Biso  1.000000 O\n   O15        1.0     0.924403      0.262517      0.081082     Biso  1.000000 O\n   O16        1.0     0.262517      0.924403      0.731998     Biso  1.000000 O\n   O17        1.0     0.081082      0.262517      0.731998     Biso  1.000000 O\n   O18        1.0     0.731998      0.924403      0.081082     Biso  1.000000 O\n   O19        1.0     0.262517      0.081082      0.924403     Biso  1.000000 O\n   O20        1.0     0.924403      0.731998      0.262517     Biso  1.000000 O\n   O21        1.0     0.262517      0.731998      0.081082     Biso  1.000000 O\n   O22        1.0     0.924403      0.081082      0.731998     Biso  1.000000 O\n   O23        1.0     0.081082      0.924403      0.262517     Biso  1.000000 O\n   O24        1.0     0.731998      0.262517      0.924403     Biso  1.000000 O\n   O25        1.0     0.203690      0.932103      0.932103     Biso  1.000000 O\n   O26        1.0     0.932103      0.203690      0.932103     Biso  1.000000 O\n   O27        1.0     0.932103      0.932103      0.203690     Biso  1.000000 O\n   O28        1.0     0.932103      0.932103      0.932103     Biso  1.000000 O\n   O29        1.0     0.846778      0.051074      0.051074     Biso  1.000000 O\n   O30        1.0     0.051074      0.846778      0.051074     Biso  1.000000 O\n   O31        1.0     0.051074      0.051074      0.846778     Biso  1.000000 O\n   O32        1.0     0.051074      0.051074      0.051074     Biso  1.000000 O\n   Zr1        1.0     0.120756      0.879244      0.120756     Biso  1.000000 Zr\n   Zr2        1.0     0.879244      0.120756      0.879244     Biso  1.000000 Zr\n   Zr3        1.0     0.120756      0.120756      0.879244     Biso  1.000000 Zr\n   Zr4        1.0     0.879244      0.879244      0.120756     Biso  1.000000 Zr\n   Zr5        1.0     0.879244      0.120756      0.120756     Biso  1.000000 Zr\n   Zr6        1.0     0.120756      0.879244      0.879244     Biso  1.000000 Zr\n",
"MOF-5": "#======================================================================\n\n# CRYSTAL DATA\n\n#----------------------------------------------------------------------\n\ndata_VESTA_phase_1\n\n\n_pd_phase_name                         'MOF-5, evacuated PBEsol               '\n_cell_length_a                         25.86584\n_cell_length_b                         25.86584\n_cell_length_c                         25.86584\n_cell_angle_alpha                      90\n_cell_angle_beta                       90\n_cell_angle_gamma                      90\n_symmetry_space_group_name_H-M         'P 1'\n_symmetry_Int_Tables_number            1\n\nloop_\n_symmetry_equiv_pos_as_xyz\n   'x, y, z'\n\nloop_\n   _atom_site_label\n   _atom_site_occupancy\n   _atom_site_fract_x\n   _atom_site_fract_y\n   _atom_site_fract_z\n   _atom_site_adp_type\n   _atom_site_B_iso_or_equiv\n   _atom_site_type_symbol\n   Zn1        1.0     0.293367      0.293367      0.206633     Biso  1.000000 Zn\n   Zn2        1.0     0.706633      0.706633      0.793367     Biso  1.000000 Zn\n   Zn3        1.0     0.706633      0.706633      0.206633     Biso  1.000000 Zn\n   Zn4        1.0     0.293367      0.293367      0.793367     Biso  1.000000 Zn\n   Zn5        1.0     0.706633      0.293367      0.793367     Biso  1.000000 Zn\n   Zn6        1.0     0.293367      0.706633      0.206633     Biso  1.000000 Zn\n   Zn7        1.0     0.293367      0.706633      0.793367     Biso  1.000000 Zn\n   Zn8        1.0     0.706633      0.293367      0.206633     Biso  1.000000 Zn\n   Zn9        1.0     0.206633      0.293367      0.293367     Biso  1.000000 Zn\n   Zn10       1.0     0.793367      0.706633      0.706633     Biso  1.000000 Zn\n   Zn11       1.0     0.206633      0.706633      0.706633     Biso  1.000000 Zn\n   Zn12       1.0     0.793367      0.293367      0.293367     Biso  1.000000 Zn\n   Zn13       1.0     0.793367      0.706633      0.293367     Biso  1.000000 Zn\n   Zn14       1.0     0.206633      0.293367      0.706633     Biso  1.000000 Zn\n   Zn15       1.0     0.793367      0.293367      0.706633     Biso  1.000000 Zn\n   Zn16       1.0     0.206633      0.706633      0.293367     Biso  1.000000 Zn\n   Zn17       1.0     0.293367      0.206633      0.293367     Biso  1.000000 Zn\n   Zn18       1.0     0.706633      0.793367      0.706633     Biso  1.000000 Zn\n   Zn19       1.0     0.706633      0.206633      0.706633     Biso  1.000000 Zn\n   Zn20       1.0     0.293367      0.793367      0.293367     Biso  1.000000 Zn\n   Zn21       1.0     0.293367      0.793367      0.706633     Biso  1.000000 Zn\n   Zn22       1.0     0.706633      0.206633      0.293367     Biso  1.000000 Zn\n   Zn23       1.0     0.706633      0.793367      0.293367     Biso  1.000000 Zn\n   Zn24       1.0     0.293367      0.206633      0.706633     Biso  1.000000 Zn\n   Zn25       1.0     0.206633      0.793367      0.793367     Biso  1.000000 Zn\n   Zn26       1.0     0.793367      0.206633      0.206633     Biso  1.000000 Zn\n   Zn27       1.0     0.206633      0.206633      0.206633     Biso  1.000000 Zn\n   Zn28       1.0     0.793367      0.793367      0.793367     Biso  1.000000 Zn\n   Zn29       1.0     0.793367      0.206633      0.793367     Biso  1.000000 Zn\n   Zn30       1.0     0.206633      0.793367      0.206633     Biso  1.000000 Zn\n   Zn31       1.0     0.793367      0.793367      0.206633     Biso  1.000000 Zn\n   Zn32       1.0     0.206633      0.206633      0.793367     Biso  1.000000 Zn\n   O1         1.0     0.250000      0.250000      0.250000     Biso  1.000000 O\n   O2         1.0     0.750000      0.750000      0.750000     Biso  1.000000 O\n   O3         1.0     0.750000      0.750000      0.250000     Biso  1.000000 O\n   O4         1.0     0.250000      0.250000      0.750000     Biso  1.000000 O\n   O5         1.0     0.750000      0.250000      0.750000     Biso  1.000000 O\n   O6         1.0     0.250000      0.750000      0.250000     Biso  1.000000 O\n   O7         1.0     0.250000      0.750000      0.750000     Biso  1.000000 O\n   O8         1.0     0.750000      0.250000      0.250000     Biso  1.000000 O\n   O9         1.0     0.281042      0.366348      0.218958     Biso  1.000000 O\n   O10        1.0     0.718958      0.633652      0.781042     Biso  1.000000 O\n   O11        1.0     0.718958      0.633652      0.218958     Biso  1.000000 O\n   O12        1.0     0.281042      0.366348      0.781042     Biso  1.000000 O\n   O13        1.0     0.718958      0.366348      0.781042     Biso  1.000000 O\n   O14        1.0     0.281042      0.633652      0.218958     Biso  1.000000 O\n   O15        1.0     0.281042      0.633652      0.781042     Biso  1.000000 O\n   O16        1.0     0.718958      0.366348      0.218958     Biso  1.000000 O\n   O17        1.0     0.218958      0.281042      0.366348     Biso  1.000000 O\n   O18        1.0     0.781042      0.718958      0.633652     Biso  1.000000 O\n   O19        1.0     0.218958      0.718958      0.633652     Biso  1.000000 O\n   O20        1.0     0.781042      0.281042      0.366348     Biso  1.000000 O\n   O21        1.0     0.781042      0.718958      0.366348     Biso  1.000000 O\n   O22        1.0     0.218958      0.281042      0.633652     Biso  1.000000 O\n   O23        1.0     0.781042      0.281042      0.633652     Biso  1.000000 O\n   O24        1.0     0.218958      0.718958      0.366348     Biso  1.000000 O\n   O25        1.0     0.366348      0.218958      0.281042     Biso  1.000000 O\n   O26        1.0     0.633652      0.781042      0.718958     Biso  1.000000 O\n   O27        1.0     0.633652      0.218958      0.718958     Biso  1.000000 O\n   O28        1.0     0.366348      0.781042      0.281042     Biso  1.000000 O\n   O29        1.0     0.366348      0.781042      0.718958     Biso  1.000000 O\n   O30        1.0     0.633652      0.218958      0.281042     Biso  1.000000 O\n   O31        1.0     0.633652      0.781042      0.281042     Biso  1.000000 O\n   O32        1.0     0.366348      0.218958      0.718958     Biso  1.000000 O\n   O33        1.0     0.366348      0.281042      0.781042     Biso  1.000000 O\n   O34        1.0     0.633652      0.718958      0.218958     Biso  1.000000 O\n   O35        1.0     0.633652      0.718958      0.781042     Biso  1.000000 O\n   O36        1.0     0.366348      0.281042      0.218958     Biso  1.000000 O\n   O37        1.0     0.366348      0.718958      0.218958     Biso  1.000000 O\n   O38        1.0     0.633652      0.281042      0.781042     Biso  1.000000 O\n   O39        1.0     0.633652      0.281042      0.218958     Biso  1.000000 O\n   O40        1.0     0.366348      0.718958      0.781042     Biso  1.000000 O\n   O41        1.0     0.281042      0.218958      0.633652     Biso  1.000000 O\n   O42        1.0     0.718958      0.781042      0.366348     Biso  1.000000 O\n   O43        1.0     0.718958      0.218958      0.366348     Biso  1.000000 O\n   O44        1.0     0.281042      0.781042      0.633652     Biso  1.000000 O\n   O45        1.0     0.718958      0.781042      0.633652     Biso  1.000000 O\n   O46        1.0     0.281042      0.218958      0.366348     Biso  1.000000 O\n   O47        1.0     0.281042      0.781042      0.366348     Biso  1.000000 O\n   O48        1.0     0.718958      0.218958      0.633652     Biso  1.000000 O\n   O49        1.0     0.218958      0.366348      0.718958     Biso  1.000000 O\n   O50        1.0     0.781042      0.633652      0.281042     Biso  1.000000 O\n   O51        1.0     0.218958      0.633652      0.281042     Biso  1.000000 O\n   O52        1.0     0.781042      0.366348      0.718958     Biso  1.000000 O\n   O53        1.0     0.781042      0.366348      0.281042     Biso  1.000000 O\n   O54        1.0     0.218958      0.633652      0.718958     Biso  1.000000 O\n   O55        1.0     0.781042      0.633652      0.718958     Biso  1.000000 O\n   O56        1.0     0.218958      0.366348      0.281042     Biso  1.000000 O\n   O57        1.0     0.281042      0.866348      0.718958     Biso  1.000000 O\n   O58        1.0     0.718958      0.133652      0.281042     Biso  1.000000 O\n   O59        1.0     0.718958      0.133652      0.718958     Biso  1.000000 O\n   O60        1.0     0.281042      0.866348      0.281042     Biso  1.000000 O\n   O61        1.0     0.718958      0.866348      0.281042     Biso  1.000000 O\n   O62        1.0     0.281042      0.133652      0.718958     Biso  1.000000 O\n   O63        1.0     0.281042      0.133652      0.281042     Biso  1.000000 O\n   O64        1.0     0.718958      0.866348      0.718958     Biso  1.000000 O\n   O65        1.0     0.218958      0.781042      0.866348     Biso  1.000000 O\n   O66        1.0     0.781042      0.218958      0.133652     Biso  1.000000 O\n   O67        1.0     0.218958      0.218958      0.133652     Biso  1.000000 O\n   O68        1.0     0.781042      0.781042      0.866348     Biso  1.000000 O\n   O69        1.0     0.781042      0.218958      0.866348     Biso  1.000000 O\n   O70        1.0     0.218958      0.781042      0.133652     Biso  1.000000 O\n   O71        1.0     0.781042      0.781042      0.133652     Biso  1.000000 O\n   O72        1.0     0.218958      0.218958      0.866348     Biso  1.000000 O\n   O73        1.0     0.281042      0.718958      0.133652     Biso  1.000000 O\n   O74        1.0     0.718958      0.281042      0.866348     Biso  1.000000 O\n   O75        1.0     0.718958      0.718958      0.866348     Biso  1.000000 O\n   O76        1.0     0.281042      0.281042      0.133652     Biso  1.000000 O\n   O77        1.0     0.718958      0.281042      0.133652     Biso  1.000000 O\n   O78        1.0     0.281042      0.718958      0.866348     Biso  1.000000 O\n   O79        1.0     0.281042      0.281042      0.866348     Biso  1.000000 O\n   O80        1.0     0.718958      0.718958      0.133652     Biso  1.000000 O\n   O81        1.0     0.218958      0.866348      0.218958     Biso  1.000000 O\n   O82        1.0     0.781042      0.133652      0.781042     Biso  1.000000 O\n   O83        1.0     0.218958      0.133652      0.781042     Biso  1.000000 O\n   O84        1.0     0.781042      0.866348      0.218958     Biso  1.000000 O\n   O85        1.0     0.781042      0.866348      0.781042     Biso  1.000000 O\n   O86        1.0     0.218958      0.133652      0.218958     Biso  1.000000 O\n   O87        1.0     0.781042      0.133652      0.218958     Biso  1.000000 O\n   O88        1.0     0.218958      0.866348      0.781042     Biso  1.000000 O\n   O89        1.0     0.866348      0.218958      0.781042     Biso  1.000000 O\n   O90        1.0     0.133652      0.781042      0.218958     Biso  1.000000 O\n   O91        1.0     0.133652      0.218958      0.218958     Biso  1.000000 O\n   O92        1.0     0.866348      0.781042      0.781042     Biso  1.000000 O\n   O93        1.0     0.866348      0.781042      0.218958     Biso  1.000000 O\n   O94        1.0     0.133652      0.218958      0.781042     Biso  1.000000 O\n   O95        1.0     0.133652      0.781042      0.781042     Biso  1.000000 O\n   O96        1.0     0.866348      0.218958      0.218958     Biso  1.000000 O\n   O97        1.0     0.866348      0.281042      0.281042     Biso  1.000000 O\n   O98        1.0     0.133652      0.718958      0.718958     Biso  1.000000 O\n   O99        1.0     0.133652      0.718958      0.281042     Biso  1.000000 O\n   O100       1.0     0.866348      0.281042      0.718958     Biso  1.000000 O\n   O101       1.0     0.866348      0.718958      0.718958     Biso  1.000000 O\n   O102       1.0     0.133652      0.281042      0.281042     Biso  1.000000 O\n   O103       1.0     0.133652      0.281042      0.718958     Biso  1.000000 O\n   O104       1.0     0.866348      0.718958      0.281042     Biso  1.000000 O\n   C1         1.0     0.250000      0.388617      0.250000     Biso  1.000000 C\n   C2         1.0     0.750000      0.611383      0.750000     Biso  1.000000 C\n   C3         1.0     0.750000      0.611383      0.250000     Biso  1.000000 C\n   C4         1.0     0.250000      0.388617      0.750000     Biso  1.000000 C\n   C5         1.0     0.750000      0.388617      0.750000     Biso  1.000000 C\n   C6         1.0     0.250000      0.611383      0.250000     Biso  1.000000 C\n   C7         1.0     0.250000      0.611383      0.750000     Biso  1.000000 C\n   C8         1.0     0.750000      0.388617      0.250000     Biso  1.000000 C\n   C9         1.0     0.250000      0.250000      0.388617     Biso  1.000000 C\n   C10        1.0     0.750000      0.750000      0.611383     Biso  1.000000 C\n   C11        1.0     0.250000      0.750000      0.611383     Biso  1.000000 C\n   C12        1.0     0.750000      0.250000      0.388617     Biso  1.000000 C\n   C13        1.0     0.750000      0.750000      0.388617     Biso  1.000000 C\n   C14        1.0     0.250000      0.250000      0.611383     Biso  1.000000 C\n   C15        1.0     0.750000      0.250000      0.611383     Biso  1.000000 C\n   C16        1.0     0.250000      0.750000      0.388617     Biso  1.000000 C\n   C17        1.0     0.388617      0.250000      0.250000     Biso  1.000000 C\n   C18        1.0     0.611383      0.750000      0.750000     Biso  1.000000 C\n   C19        1.0     0.611383      0.250000      0.750000     Biso  1.000000 C\n   C20        1.0     0.388617      0.750000      0.250000     Biso  1.000000 C\n   C21        1.0     0.388617      0.750000      0.750000     Biso  1.000000 C\n   C22        1.0     0.611383      0.250000      0.250000     Biso  1.000000 C\n   C23        1.0     0.611383      0.750000      0.250000     Biso  1.000000 C\n   C24        1.0     0.388617      0.250000      0.750000     Biso  1.000000 C\n   C25        1.0     0.250000      0.888617      0.750000     Biso  1.000000 C\n   C26        1.0     0.750000      0.111383      0.250000     Biso  1.000000 C\n   C27        1.0     0.750000      0.111383      0.750000     Biso  1.000000 C\n   C28        1.0     0.250000      0.888617      0.250000     Biso  1.000000 C\n   C29        1.0     0.750000      0.888617      0.250000     Biso  1.000000 C\n   C30        1.0     0.250000      0.111383      0.750000     Biso  1.000000 C\n   C31        1.0     0.250000      0.111383      0.250000     Biso  1.000000 C\n   C32        1.0     0.750000      0.888617      0.750000     Biso  1.000000 C\n   C33        1.0     0.250000      0.750000      0.888617     Biso  1.000000 C\n   C34        1.0     0.750000      0.250000      0.111383     Biso  1.000000 C\n   C35        1.0     0.250000      0.250000      0.111383     Biso  1.000000 C\n   C36        1.0     0.750000      0.750000      0.888617     Biso  1.000000 C\n   C37        1.0     0.750000      0.250000      0.888617     Biso  1.000000 C\n   C38        1.0     0.250000      0.750000      0.111383     Biso  1.000000 C\n   C39        1.0     0.750000      0.750000      0.111383     Biso  1.000000 C\n   C40        1.0     0.250000      0.250000      0.888617     Biso  1.000000 C\n   C41        1.0     0.888617      0.250000      0.750000     Biso  1.000000 C\n   C42        1.0     0.111383      0.750000      0.250000     Biso  1.000000 C\n   C43        1.0     0.111383      0.250000      0.250000     Biso  1.000000 C\n   C44        1.0     0.888617      0.750000      0.750000     Biso  1.000000 C\n   C45        1.0     0.888617      0.750000      0.250000     Biso  1.000000 C\n   C46        1.0     0.111383      0.250000      0.750000     Biso  1.000000 C\n   C47        1.0     0.111383      0.750000      0.750000     Biso  1.000000 C\n   C48        1.0     0.888617      0.250000      0.250000     Biso  1.000000 C\n   C49        1.0     0.250000      0.446192      0.250000     Biso  1.000000 C\n   C50        1.0     0.750000      0.553808      0.750000     Biso  1.000000 C\n   C51        1.0     0.750000      0.553808      0.250000     Biso  1.000000 C\n   C52        1.0     0.250000      0.446192      0.750000     Biso  1.000000 C\n   C53        1.0     0.750000      0.446192      0.750000     Biso  1.000000 C\n   C54        1.0     0.250000      0.553808      0.250000     Biso  1.000000 C\n   C55        1.0     0.250000      0.553808      0.750000     Biso  1.000000 C\n   C56        1.0     0.750000      0.446192      0.250000     Biso  1.000000 C\n   C57        1.0     0.250000      0.250000      0.446192     Biso  1.000000 C\n   C58        1.0     0.750000      0.750000      0.553808     Biso  1.000000 C\n   C59        1.0     0.250000      0.750000      0.553808     Biso  1.000000 C\n   C60        1.0     0.750000      0.250000      0.446192     Biso  1.000000 C\n   C61        1.0     0.750000      0.750000      0.446192     Biso  1.000000 C\n   C62        1.0     0.250000      0.250000      0.553808     Biso  1.000000 C\n   C63        1.0     0.750000      0.250000      0.553808     Biso  1.000000 C\n   C64        1.0     0.250000      0.750000      0.446192     Biso  1.000000 C\n   C65        1.0     0.446192      0.250000      0.250000     Biso  1.000000 C\n   C66        1.0     0.553808      0.750000      0.750000     Biso  1.000000 C\n   C67        1.0     0.553808      0.250000      0.750000     Biso  1.000000 C\n   C68        1.0     0.446192      0.750000      0.250000     Biso  1.000000 C\n   C69        1.0     0.446192      0.750000      0.750000     Biso  1.000000 C\n   C70        1.0     0.553808      0.250000      0.250000     Biso  1.000000 C\n   C71        1.0     0.553808      0.750000      0.250000     Biso  1.000000 C\n   C72        1.0     0.446192      0.250000      0.750000     Biso  1.000000 C\n   C73        1.0     0.250000      0.946192      0.750000     Biso  1.000000 C\n   C74        1.0     0.750000      0.053808      0.250000     Biso  1.000000 C\n   C75        1.0     0.750000      0.053808      0.750000     Biso  1.000000 C\n   C76        1.0     0.250000      0.946192      0.250000     Biso  1.000000 C\n   C77        1.0     0.750000      0.946192      0.250000     Biso  1.000000 C\n   C78        1.0     0.250000      0.053808      0.750000     Biso  1.000000 C\n   C79        1.0     0.250000      0.053808      0.250000     Biso  1.000000 C\n   C80        1.0     0.750000      0.946192      0.750000     Biso  1.000000 C\n   C81        1.0     0.250000      0.750000      0.946192     Biso  1.000000 C\n   C82        1.0     0.750000      0.250000      0.053808     Biso  1.000000 C\n   C83        1.0     0.250000      0.250000      0.053808     Biso  1.000000 C\n   C84        1.0     0.750000      0.750000      0.946192     Biso  1.000000 C\n   C85        1.0     0.750000      0.250000      0.946192     Biso  1.000000 C\n   C86        1.0     0.250000      0.750000      0.053808     Biso  1.000000 C\n   C87        1.0     0.750000      0.750000      0.053808     Biso  1.000000 C\n   C88        1.0     0.250000      0.250000      0.946192     Biso  1.000000 C\n   C89        1.0     0.946192      0.250000      0.750000     Biso  1.000000 C\n   C90        1.0     0.053808      0.750000      0.250000     Biso  1.000000 C\n   C91        1.0     0.053808      0.250000      0.250000     Biso  1.000000 C\n   C92        1.0     0.946192      0.750000      0.750000     Biso  1.000000 C\n   C93        1.0     0.946192      0.750000      0.250000     Biso  1.000000 C\n   C94        1.0     0.053808      0.250000      0.750000     Biso  1.000000 C\n   C95        1.0     0.053808      0.750000      0.750000     Biso  1.000000 C\n   C96        1.0     0.946192      0.250000      0.250000     Biso  1.000000 C\n   C97        1.0     0.283167      0.473179      0.216833     Biso  1.000000 C\n   C98        1.0     0.716833      0.526821      0.783167     Biso  1.000000 C\n   C99        1.0     0.716833      0.526821      0.216833     Biso  1.000000 C\n   C100       1.0     0.283167      0.473179      0.783167     Biso  1.000000 C\n   C101       1.0     0.716833      0.473179      0.783167     Biso  1.000000 C\n   C102       1.0     0.283167      0.526821      0.216833     Biso  1.000000 C\n   C103       1.0     0.283167      0.526821      0.783167     Biso  1.000000 C\n   C104       1.0     0.716833      0.473179      0.216833     Biso  1.000000 C\n   C105       1.0     0.216833      0.283167      0.473179     Biso  1.000000 C\n   C106       1.0     0.783167      0.716833      0.526821     Biso  1.000000 C\n   C107       1.0     0.216833      0.716833      0.526821     Biso  1.000000 C\n   C108       1.0     0.783167      0.283167      0.473179     Biso  1.000000 C\n   C109       1.0     0.783167      0.716833      0.473179     Biso  1.000000 C\n   C110       1.0     0.216833      0.283167      0.526821     Biso  1.000000 C\n   C111       1.0     0.783167      0.283167      0.526821     Biso  1.000000 C\n   C112       1.0     0.216833      0.716833      0.473179     Biso  1.000000 C\n   C113       1.0     0.473179      0.216833      0.283167     Biso  1.000000 C\n   C114       1.0     0.526821      0.783167      0.716833     Biso  1.000000 C\n   C115       1.0     0.526821      0.216833      0.716833     Biso  1.000000 C\n   C116       1.0     0.473179      0.783167      0.283167     Biso  1.000000 C\n   C117       1.0     0.473179      0.783167      0.716833     Biso  1.000000 C\n   C118       1.0     0.526821      0.216833      0.283167     Biso  1.000000 C\n   C119       1.0     0.526821      0.783167      0.283167     Biso  1.000000 C\n   C120       1.0     0.473179      0.216833      0.716833     Biso  1.000000 C\n   C121       1.0     0.473179      0.283167      0.783167     Biso  1.000000 C\n   C122       1.0     0.526821      0.716833      0.216833     Biso  1.000000 C\n   C123       1.0     0.526821      0.716833      0.783167     Biso  1.000000 C\n   C124       1.0     0.473179      0.283167      0.216833     Biso  1.000000 C\n   C125       1.0     0.473179      0.716833      0.216833     Biso  1.000000 C\n   C126       1.0     0.526821      0.283167      0.783167     Biso  1.000000 C\n   C127       1.0     0.526821      0.283167      0.216833     Biso  1.000000 C\n   C128       1.0     0.473179      0.716833      0.783167     Biso  1.000000 C\n   C129       1.0     0.283167      0.216833      0.526821     Biso  1.000000 C\n   C130       1.0     0.716833      0.783167      0.473179     Biso  1.000000 C\n   C131       1.0     0.716833      0.216833      0.473179     Biso  1.000000 C\n   C132       1.0     0.283167      0.783167      0.526821     Biso  1.000000 C\n   C133       1.0     0.716833      0.783167      0.526821     Biso  1.000000 C\n   C134       1.0     0.283167      0.216833      0.473179     Biso  1.000000 C\n   C135       1.0     0.283167      0.783167      0.473179     Biso  1.000000 C\n   C136       1.0     0.716833      0.216833      0.526821     Biso  1.000000 C\n   C137       1.0     0.216833      0.473179      0.716833     Biso  1.000000 C\n   C138       1.0     0.783167      0.526821      0.283167     Biso  1.000000 C\n   C139       1.0     0.216833      0.526821      0.283167     Biso  1.000000 C\n   C140       1.0     0.783167      0.473179      0.716833     Biso  1.000000 C\n   C141       1.0     0.783167      0.473179      0.283167     Biso  1.000000 C\n   C142       1.0     0.216833      0.526821      0.716833     Biso  1.000000 C\n   C143       1.0     0.783167      0.526821      0.716833     Biso  1.000000 C\n   C144       1.0     0.216833      0.473179      0.283167     Biso  1.000000 C\n   C145       1.0     0.283167      0.973179      0.716833     Biso  1.000000 C\n   C146       1.0     0.716833      0.026821      0.283167     Biso  1.000000 C\n   C147       1.0     0.716833      0.026821      0.716833     Biso  1.000000 C\n   C148       1.0     0.283167      0.973179      0.283167     Biso  1.000000 C\n   C149       1.0     0.716833      0.973179      0.283167     Biso  1.000000 C\n   C150       1.0     0.283167      0.026821      0.716833     Biso  1.000000 C\n   C151       1.0     0.283167      0.026821      0.283167     Biso  1.000000 C\n   C152       1.0     0.716833      0.973179      0.716833     Biso  1.000000 C\n   C153       1.0     0.216833      0.783167      0.973179     Biso  1.000000 C\n   C154       1.0     0.783167      0.216833      0.026821     Biso  1.000000 C\n   C155       1.0     0.216833      0.216833      0.026821     Biso  1.000000 C\n   C156       1.0     0.783167      0.783167      0.973179     Biso  1.000000 C\n   C157       1.0     0.783167      0.216833      0.973179     Biso  1.000000 C\n   C158       1.0     0.216833      0.783167      0.026821     Biso  1.000000 C\n   C159       1.0     0.783167      0.783167      0.026821     Biso  1.000000 C\n   C160       1.0     0.216833      0.216833      0.973179     Biso  1.000000 C\n   C161       1.0     0.283167      0.716833      0.026821     Biso  1.000000 C\n   C162       1.0     0.716833      0.283167      0.973179     Biso  1.000000 C\n   C163       1.0     0.716833      0.716833      0.973179     Biso  1.000000 C\n   C164       1.0     0.283167      0.283167      0.026821     Biso  1.000000 C\n   C165       1.0     0.716833      0.283167      0.026821     Biso  1.000000 C\n   C166       1.0     0.283167      0.716833      0.973179     Biso  1.000000 C\n   C167       1.0     0.283167      0.283167      0.973179     Biso  1.000000 C\n   C168       1.0     0.716833      0.716833      0.026821     Biso  1.000000 C\n   C169       1.0     0.216833      0.973179      0.216833     Biso  1.000000 C\n   C170       1.0     0.783167      0.026821      0.783167     Biso  1.000000 C\n   C171       1.0     0.216833      0.026821      0.783167     Biso  1.000000 C\n   C172       1.0     0.783167      0.973179      0.216833     Biso  1.000000 C\n   C173       1.0     0.783167      0.973179      0.783167     Biso  1.000000 C\n   C174       1.0     0.216833      0.026821      0.216833     Biso  1.000000 C\n   C175       1.0     0.783167      0.026821      0.216833     Biso  1.000000 C\n   C176       1.0     0.216833      0.973179      0.783167     Biso  1.000000 C\n   C177       1.0     0.973179      0.216833      0.783167     Biso  1.000000 C\n   C178       1.0     0.026821      0.783167      0.216833     Biso  1.000000 C\n   C179       1.0     0.026821      0.216833      0.216833     Biso  1.000000 C\n   C180       1.0     0.973179      0.783167      0.783167     Biso  1.000000 C\n   C181       1.0     0.973179      0.783167      0.216833     Biso  1.000000 C\n   C182       1.0     0.026821      0.216833      0.783167     Biso  1.000000 C\n   C183       1.0     0.026821      0.783167      0.783167     Biso  1.000000 C\n   C184       1.0     0.973179      0.216833      0.216833     Biso  1.000000 C\n   C185       1.0     0.973179      0.283167      0.283167     Biso  1.000000 C\n   C186       1.0     0.026821      0.716833      0.716833     Biso  1.000000 C\n   C187       1.0     0.026821      0.716833      0.283167     Biso  1.000000 C\n   C188       1.0     0.973179      0.283167      0.716833     Biso  1.000000 C\n   C189       1.0     0.973179      0.716833      0.716833     Biso  1.000000 C\n   C190       1.0     0.026821      0.283167      0.283167     Biso  1.000000 C\n   C191       1.0     0.026821      0.283167      0.716833     Biso  1.000000 C\n   C192       1.0     0.973179      0.716833      0.283167     Biso  1.000000 C\n   H1         1.0     0.308690      0.451190      0.191310     Biso  1.000000 H\n   H2         1.0     0.691310      0.548810      0.808690     Biso  1.000000 H\n   H3         1.0     0.691310      0.548810      0.191310     Biso  1.000000 H\n   H4         1.0     0.308690      0.451190      0.808690     Biso  1.000000 H\n   H5         1.0     0.691310      0.451190      0.808690     Biso  1.000000 H\n   H6         1.0     0.308690      0.548810      0.191310     Biso  1.000000 H\n   H7         1.0     0.308690      0.548810      0.808690     Biso  1.000000 H\n   H8         1.0     0.691310      0.451190      0.191310     Biso  1.000000 H\n   H9         1.0     0.191310      0.308690      0.451190     Biso  1.000000 H\n   H10        1.0     0.808690      0.691310      0.548810     Biso  1.000000 H\n   H11        1.0     0.191310      0.691310      0.548810     Biso  1.000000 H\n   H12        1.0     0.808690      0.308690      0.451190     Biso  1.000000 H\n   H13        1.0     0.808690      0.691310      0.451190     Biso  1.000000 H\n   H14        1.0     0.191310      0.308690      0.548810     Biso  1.000000 H\n   H15        1.0     0.808690      0.308690      0.548810     Biso  1.000000 H\n   H16        1.0     0.191310      0.691310      0.451190     Biso  1.000000 H\n   H17        1.0     0.451190      0.191310      0.308690     Biso  1.000000 H\n   H18        1.0     0.548810      0.808690      0.691310     Biso  1.000000 H\n   H19        1.0     0.548810      0.191310      0.691310     Biso  1.000000 H\n   H20        1.0     0.451190      0.808690      0.308690     Biso  1.000000 H\n   H21        1.0     0.451190      0.808690      0.691310     Biso  1.000000 H\n   H22        1.0     0.548810      0.191310      0.308690     Biso  1.000000 H\n   H23        1.0     0.548810      0.808690      0.308690     Biso  1.000000 H\n   H24        1.0     0.451190      0.191310      0.691310     Biso  1.000000 H\n   H25        1.0     0.451190      0.308690      0.808690     Biso  1.000000 H\n   H26        1.0     0.548810      0.691310      0.191310     Biso  1.000000 H\n   H27        1.0     0.548810      0.691310      0.808690     Biso  1.000000 H\n   H28        1.0     0.451190      0.308690      0.191310     Biso  1.000000 H\n   H29        1.0     0.451190      0.691310      0.191310     Biso  1.000000 H\n   H30        1.0     0.548810      0.308690      0.808690     Biso  1.000000 H\n   H31        1.0     0.548810      0.308690      0.191310     Biso  1.000000 H\n   H32        1.0     0.451190      0.691310      0.808690     Biso  1.000000 H\n   H33        1.0     0.308690      0.191310      0.548810     Biso  1.000000 H\n   H34        1.0     0.691310      0.808690      0.451190     Biso  1.000000 H\n   H35        1.0     0.691310      0.191310      0.451190     Biso  1.000000 H\n   H36        1.0     0.308690      0.808690      0.548810     Biso  1.000000 H\n   H37        1.0     0.691310      0.808690      0.548810     Biso  1.000000 H\n   H38        1.0     0.308690      0.191310      0.451190     Biso  1.000000 H\n   H39        1.0     0.308690      0.808690      0.451190     Biso  1.000000 H\n   H40        1.0     0.691310      0.191310      0.548810     Biso  1.000000 H\n   H41        1.0     0.191310      0.451190      0.691310     Biso  1.000000 H\n   H42        1.0     0.808690      0.548810      0.308690     Biso  1.000000 H\n   H43        1.0     0.191310      0.548810      0.308690     Biso  1.000000 H\n   H44        1.0     0.808690      0.451190      0.691310     Biso  1.000000 H\n   H45        1.0     0.808690      0.451190      0.308690     Biso  1.000000 H\n   H46        1.0     0.191310      0.548810      0.691310     Biso  1.000000 H\n   H47        1.0     0.808690      0.548810      0.691310     Biso  1.000000 H\n   H48        1.0     0.191310      0.451190      0.308690     Biso  1.000000 H\n   H49        1.0     0.308690      0.951190      0.691310     Biso  1.000000 H\n   H50        1.0     0.691310      0.048810      0.308690     Biso  1.000000 H\n   H51        1.0     0.691310      0.048810      0.691310     Biso  1.000000 H\n   H52        1.0     0.308690      0.951190      0.308690     Biso  1.000000 H\n   H53        1.0     0.691310      0.951190      0.308690     Biso  1.000000 H\n   H54        1.0     0.308690      0.048810      0.691310     Biso  1.000000 H\n   H55        1.0     0.308690      0.048810      0.308690     Biso  1.000000 H\n   H56        1.0     0.691310      0.951190      0.691310     Biso  1.000000 H\n   H57        1.0     0.191310      0.808690      0.951190     Biso  1.000000 H\n   H58        1.0     0.808690      0.191310      0.048810     Biso  1.000000 H\n   H59        1.0     0.191310      0.191310      0.048810     Biso  1.000000 H\n   H60        1.0     0.808690      0.808690      0.951190     Biso  1.000000 H\n   H61        1.0     0.808690      0.191310      0.951190     Biso  1.000000 H\n   H62        1.0     0.191310      0.808690      0.048810     Biso  1.000000 H\n   H63        1.0     0.808690      0.808690      0.048810     Biso  1.000000 H\n   H64        1.0     0.191310      0.191310      0.951190     Biso  1.000000 H\n   H65        1.0     0.308690      0.691310      0.048810     Biso  1.000000 H\n   H66        1.0     0.691310      0.308690      0.951190     Biso  1.000000 H\n   H67        1.0     0.691310      0.691310      0.951190     Biso  1.000000 H\n   H68        1.0     0.308690      0.308690      0.048810     Biso  1.000000 H\n   H69        1.0     0.691310      0.308690      0.048810     Biso  1.000000 H\n   H70        1.0     0.308690      0.691310      0.951190     Biso  1.000000 H\n   H71        1.0     0.308690      0.308690      0.951190     Biso  1.000000 H\n   H72        1.0     0.691310      0.691310      0.048810     Biso  1.000000 H\n   H73        1.0     0.191310      0.951190      0.191310     Biso  1.000000 H\n   H74        1.0     0.808690      0.048810      0.808690     Biso  1.000000 H\n   H75        1.0     0.191310      0.048810      0.808690     Biso  1.000000 H\n   H76        1.0     0.808690      0.951190      0.191310     Biso  1.000000 H\n   H77        1.0     0.808690      0.951190      0.808690     Biso  1.000000 H\n   H78        1.0     0.191310      0.048810      0.191310     Biso  1.000000 H\n   H79        1.0     0.808690      0.048810      0.191310     Biso  1.000000 H\n   H80        1.0     0.191310      0.951190      0.808690     Biso  1.000000 H\n   H81        1.0     0.951190      0.191310      0.808690     Biso  1.000000 H\n   H82        1.0     0.048810      0.808690      0.191310     Biso  1.000000 H\n   H83        1.0     0.048810      0.191310      0.191310     Biso  1.000000 H\n   H84        1.0     0.951190      0.808690      0.808690     Biso  1.000000 H\n   H85        1.0     0.951190      0.808690      0.191310     Biso  1.000000 H\n   H86        1.0     0.048810      0.191310      0.808690     Biso  1.000000 H\n   H87        1.0     0.048810      0.808690      0.808690     Biso  1.000000 H\n   H88        1.0     0.951190      0.191310      0.191310     Biso  1.000000 H\n   H89        1.0     0.951190      0.308690      0.308690     Biso  1.000000 H\n   H90        1.0     0.048810      0.691310      0.691310     Biso  1.000000 H\n   H91        1.0     0.048810      0.691310      0.308690     Biso  1.000000 H\n   H92        1.0     0.951190      0.308690      0.691310     Biso  1.000000 H\n   H93        1.0     0.951190      0.691310      0.691310     Biso  1.000000 H\n   H94        1.0     0.048810      0.308690      0.308690     Biso  1.000000 H\n   H95        1.0     0.048810      0.308690      0.691310     Biso  1.000000 H\n   H96        1.0     0.951190      0.691310      0.308690     Biso  1.000000 H\n"
}
from pathlib import Path
for name, text in CIFS.items():
    Path(f'{name}.cif').write_text(text, encoding='utf-8')
print('Wrote', ', '.join(f'{n}.cif' for n in CIFS))


In [ ]:
# Science code: original pipeline cell plus the changes marked SYMMETRIC-SLAB CHANGE.
MAIN_SRC = "# MorphoGraph-MOF — revised, diagnostic, atomistic surface/morphology pipeline\n# Intended for Google Colab. Run the install cell from the companion notebook first.\n#\n# Scientific design:\n# 1) Use the primitive-standard cell where possible to avoid unnecessary supercells.\n# 2) Relax the bulk at fixed cell and use its energy per atom as the reference.\n# 3) Generate directed Miller-orientation representatives using the actual point group.\n# 4) Relax stoichiometric slabs, including asymmetric slabs.\n#    SYMMETRIC-SLAB VERSION: symmetrized slabs are added; see \"SYMMETRIC-SLAB CHANGE\" comments.\n# 5) For each facet family, fit E_slab - N E_bulk = A*(gamma_top + gamma_bottom).\n#    Individual termination energies are accepted only if the equations are full-rank,\n#    the fit residual meets the user-entered tolerance, and fitted gammas are positive.\n# 6) Construct a signed-normal Wulff polyhedron only if every requested directed facet\n#    class has a validated surface-energy estimate. No missing facet energies are invented.\n# 7) Also construct clearly labelled geometry-only controls (equal-gamma and BFDH-like).\n# 8) Clip repeated atoms from the actual periodic CIF inside each polyhedron for visualization.\n#\n# Important: this is a computational workflow, not a guarantee that CHGNet is accurate for\n# every MOF. Check convergence, slab thickness/vacuum sensitivity, and DFT validation before\n# scientific publication. Disordered CIFs are rejected rather than silently altered.\n\nimport os\nimport re\nimport json\nimport math\nimport hashlib\nimport shutil\nfrom datetime import datetime\nimport inspect\nimport itertools\nimport warnings\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nfrom contextlib import redirect_stdout, redirect_stderr\nfrom io import StringIO\nfrom zipfile import ZipFile, ZIP_DEFLATED\n\nimport numpy as np\nimport pandas as pd\nfrom IPython.display import display, Markdown\n\nfrom pymatgen.core import Structure, Lattice, Molecule\nfrom pymatgen.core.surface import SlabGenerator\nfrom pymatgen.symmetry.analyzer import SpacegroupAnalyzer\nfrom pymatgen.analysis.local_env import CrystalNN\nfrom chgnet.model import CHGNet\nfrom chgnet.model.dynamics import StructOptimizer\nfrom scipy.spatial import ConvexHull, HalfspaceIntersection, QhullError\n\nwarnings.filterwarnings(\"once\", category=DeprecationWarning)\n\n# -----------------------------------------------------------------------------\n# 1. User-visible parameters. Enter accepts the shown default.\n# -----------------------------------------------------------------------------\ndef ask_int(prompt, default, minimum=1):\n    while True:\n        raw = input(f\"{prompt} [{default}]: \").strip()\n        try:\n            value = int(raw) if raw else int(default)\n            if value < minimum:\n                raise ValueError(f\"must be >= {minimum}\")\n            return value\n        except Exception as exc:\n            print(f\"Invalid value ({exc}). Try again.\")\n\n\ndef ask_float(prompt, default, minimum=0.0):\n    while True:\n        raw = input(f\"{prompt} [{default}]: \").strip()\n        try:\n            value = float(raw) if raw else float(default)\n            if not np.isfinite(value) or value <= minimum:\n                raise ValueError(f\"must be finite and > {minimum}\")\n            return value\n        except Exception as exc:\n            print(f\"Invalid value ({exc}). Try again.\")\n\n\nMAX_MILLER_INDEX = SETTINGS[\"MAX_MILLER_INDEX\"]\nSLAB_THICKNESS_A = SETTINGS[\"SLAB_THICKNESS_A\"]\nVACUUM_A = SETTINGS[\"VACUUM_A\"]\nFORCE_TOL_EV_A = SETTINGS[\"FORCE_TOL_EV_A\"]\nMAX_RELAX_STEPS = SETTINGS[\"MAX_RELAX_STEPS\"]\nPARTICLE_RADIUS_A = SETTINGS[\"PARTICLE_RADIUS_A\"]\nDISPLAY_REPEATS = SETTINGS[\"DISPLAY_REPEATS\"]\nSYMPREC_A = SETTINGS[\"SYMPREC_A\"]\nSURFACE_FINGERPRINT_DEPTH_A = SETTINGS[\"SURFACE_FINGERPRINT_DEPTH_A\"]\nFINGERPRINT_DISTANCE_TOL_A = SETTINGS[\"FINGERPRINT_DISTANCE_TOL_A\"]\nPAIR_FIT_RMS_TOL_EV_A2 = SETTINGS[\"PAIR_FIT_RMS_TOL_EV_A2\"]\nTERMINATION_SHIFT_TOL_FRACTIONAL = SETTINGS[\"TERMINATION_SHIFT_TOL_FRACTIONAL\"]\n# SYMMETRIC-SLAB CHANGE: two added settings.\nSYMMETRIC_SLABS = bool(SETTINGS.get(\"SYMMETRIC_SLABS\", True))\nPAIR_AVERAGE_FALLBACK = bool(SETTINGS.get(\"PAIR_AVERAGE_FALLBACK\", True))\nKEEP_LINKERS_INTACT = bool(SETTINGS.get(\"KEEP_LINKERS_INTACT\", True))\n# Organic bonds a MOF surface should not cut (A): breaking them leaves loose H, CH, C2H2 or CO2\n# pieces in the slab, which the first runs showed in most slabs. Metal-ligand bonds may be cut.\nLINKER_BONDS = {(\"C\", \"C\"): 1.75, (\"C\", \"H\"): 1.25, (\"C\", \"O\"): 1.6, (\"C\", \"N\"): 1.6,\n                (\"N\", \"N\"): 1.6, (\"N\", \"H\"): 1.2, (\"O\", \"H\"): 1.2}\n\n# -----------------------------------------------------------------------------\n# 2. Load CIF and obtain a primitive-standard working cell if possible.\n# Miller indices in this run are referenced to the recorded working cell.\n# -----------------------------------------------------------------------------\ntry:\n    CIF_PATH\nexcept NameError:\n    try:\n        from google.colab import files\n        uploaded = files.upload()\n        cif_names = [name for name in uploaded if name.lower().endswith(\".cif\")]\n        if not cif_names:\n            raise ValueError(\"No .cif file was uploaded. Upload an ordered MOF CIF.\")\n        CIF_PATH = cif_names[0]\n    except ImportError:\n        CIF_PATH = input(\"Path to ordered MOF CIF: \").strip()\n\nCIF_PATH = str(CIF_PATH)\nif not os.path.isfile(CIF_PATH):\n    raise FileNotFoundError(f\"CIF not found: {CIF_PATH}\")\n\ninput_structure = Structure.from_file(CIF_PATH)\nif len(input_structure) == 0:\n    raise ValueError(\"The CIF parser returned an empty structure.\")\nif not input_structure.is_ordered:\n    raise ValueError(\n        \"This CIF has partial/disordered occupancies. Make and validate an ordered atomic model first; \"\n        \"the pipeline will not silently choose one atom from a mixed-occupancy site.\"\n    )\n\nstandardization_notes = []\ntry:\n    _sga = SpacegroupAnalyzer(input_structure, symprec=SYMPREC_A)\n    _primitive = _sga.get_primitive_standard_structure()\n    if _primitive is not None and len(_primitive) > 0 and _primitive.is_ordered:\n        working_structure = _primitive\n        standardization_notes.append(\"Using primitive-standard structure to reduce redundant atoms in slab calculations.\")\n    else:\n        working_structure = input_structure.copy()\n        standardization_notes.append(\"Primitive standardization returned no suitable structure; using input CIF cell.\")\nexcept Exception as exc:\n    working_structure = input_structure.copy()\n    standardization_notes.append(f\"Primitive standardization failed; using input CIF cell ({type(exc).__name__}: {exc}).\")\n\nformula = working_structure.composition.reduced_formula\nsafe_name = re.sub(r\"[^A-Za-z0-9_.-]+\", \"_\", Path(CIF_PATH).stem).strip(\"_\") or \"MOF\"\nRUN_ID = datetime.now().strftime(\"%Y%m%d_%H%M%S\")\nRESULTS_DIR = Path(f\"MorphoGraph_MOF_results_{safe_name}_{RUN_ID}\")\nINITIAL_SLAB_DIR = RESULTS_DIR / \"slabs_initial_unique\"\nRELAXED_SLAB_DIR = RESULTS_DIR / \"slabs_relaxed\"\nPARTICLE_DIR = RESULTS_DIR / \"particles\"\nfor _dir in (RESULTS_DIR, INITIAL_SLAB_DIR, RELAXED_SLAB_DIR, PARTICLE_DIR):\n    _dir.mkdir(parents=True, exist_ok=True)\nworking_structure.to(filename=str(RESULTS_DIR / f\"{safe_name}_working_cell.cif\"))\nprint(\"\\n================ MOF INPUT ================\")\nprint(f\"Input file       : {CIF_PATH}\")\nprint(f\"Formula          : {formula}\")\nprint(f\"Input atoms      : {len(input_structure)}\")\nprint(f\"Working-cell atoms: {len(working_structure)}\")\nprint(f\"Lattice lengths (A): {np.round(working_structure.lattice.abc, 5)}\")\nprint(f\"Lattice angles (deg): {np.round(working_structure.lattice.angles, 4)}\")\nprint(f\"Cell volume (A^3): {working_structure.volume:.5f}\")\nfor note in standardization_notes:\n    print(\"Note:\", note)\nprint(\"Miller indices will be referenced to the working-cell setting written to the output CIF.\")\nprint(\"===========================================\\n\")\n\n# -----------------------------------------------------------------------------\n# 3. General utilities: formulas, energies, Miller orbits, and slab fingerprints.\n# -----------------------------------------------------------------------------\ndef normalize_hkl(hkl):\n    vals = tuple(int(round(v)) for v in hkl)\n    if vals == (0, 0, 0):\n        return None\n    g = math.gcd(math.gcd(abs(vals[0]), abs(vals[1])), abs(vals[2]))\n    if g > 1:\n        vals = tuple(v // g for v in vals)\n    return vals\n\n\ndef enumerate_primitive_directed_hkls(max_index):\n    values = set()\n    for raw in itertools.product(range(-max_index, max_index + 1), repeat=3):\n        hkl = normalize_hkl(raw)\n        if hkl is not None and max(abs(v) for v in hkl) <= max_index:\n            values.add(hkl)\n    return sorted(values)\n\n\ndef directed_miller_orbits(structure, max_index, symprec):\n    \"\"\"Reduce signed Miller indices only by actual point-group operations (not by ± equivalence).\"\"\"\n    all_hkls = set(enumerate_primitive_directed_hkls(max_index))\n    notes = []\n    reciprocal_ops = []\n    try:\n        analyzer = SpacegroupAnalyzer(structure, symprec=symprec)\n        operations = analyzer.get_point_group_operations(cartesian=False)\n        for op in operations:\n            rot = np.asarray(op.rotation_matrix, dtype=float)\n            transform = np.linalg.inv(rot).T\n            if not np.allclose(transform, np.rint(transform), atol=1e-5):\n                notes.append(\"Skipped a non-integer reciprocal rotation in this cell setting; it was not used to merge facets.\")\n                continue\n            reciprocal_ops.append(np.rint(transform).astype(int))\n    except Exception as exc:\n        notes.append(f\"Point-group reduction failed ({type(exc).__name__}: {exc}); use identity-only facet orbits.\")\n    if not reciprocal_ops:\n        reciprocal_ops = [np.eye(3, dtype=int)]\n\n    unseen = set(all_hkls)\n    representatives = []\n    orbit_map = {}\n    direction_to_rep = {}\n    while unseen:\n        seed = min(unseen)\n        orbit = set()\n        frontier = [seed]\n        while frontier:\n            current = frontier.pop()\n            if current in orbit:\n                continue\n            orbit.add(current)\n            for transform in reciprocal_ops:\n                mapped_arr = transform @ np.asarray(current, dtype=int)\n                mapped = normalize_hkl(mapped_arr)\n                if mapped is None or max(abs(v) for v in mapped) > max_index or mapped not in all_hkls:\n                    continue\n                if mapped not in orbit:\n                    frontier.append(mapped)\n        # Ensure an operation with numerical setting issues never removes unaccounted indices.\n        orbit &= all_hkls\n        if not orbit:\n            orbit = {seed}\n        rep = min(orbit)\n        representatives.append(rep)\n        orbit_map[rep] = sorted(orbit)\n        for direction in orbit:\n            direction_to_rep[direction] = rep\n        unseen -= orbit\n    representatives = sorted(set(representatives))\n    return representatives, orbit_map, direction_to_rep, sorted(all_hkls), notes\n\n\ndef element_amounts(structure):\n    return {str(k): float(v) for k, v in structure.composition.get_el_amt_dict().items()}\n\n\ndef composition_is_proportional(candidate, bulk, rtol=1e-5, atol=1e-5):\n    a = element_amounts(candidate)\n    b = element_amounts(bulk)\n    if not b or set(a) != set(b):\n        return False\n    ratios = [a[e] / b[e] for e in b]\n    return bool(ratios) and all(np.isclose(r, ratios[0], rtol=rtol, atol=atol) for r in ratios)\n\n\ndef surface_area_A2(structure):\n    matrix = np.asarray(structure.lattice.matrix, dtype=float)\n    # SlabGenerator reorients the lattice so rows 0 and 1 span the surface plane.\n    area = float(np.linalg.norm(np.cross(matrix[0], matrix[1])))\n    if not np.isfinite(area) or area <= 0:\n        raise ValueError(f\"Invalid in-plane slab area: {area}\")\n    return area\n\n\ndef to_numpy(value):\n    if hasattr(value, \"detach\"):\n        value = value.detach().cpu().numpy()\n    return np.asarray(value, dtype=float)\n\n\ndef model_energy_force(model, structure):\n    \"\"\"Return total energy (eV), energy/atom (eV/atom), and maximum force (eV/A).\"\"\"\n    pred = model.predict_structure(structure, task=\"ef\")\n    e = to_numpy(pred[\"e\"]).reshape(-1)\n    if not len(e) or not np.isfinite(e[0]):\n        raise ValueError(\"CHGNet returned a non-finite energy.\")\n    forces = to_numpy(pred[\"f\"])\n    if forces.shape != (len(structure), 3) or not np.isfinite(forces).all():\n        raise ValueError(f\"Unexpected or non-finite force array: {forces.shape}\")\n    epa = float(e[0])\n    fmax = float(np.max(np.linalg.norm(forces, axis=1))) if len(forces) else 0.0\n    return epa * len(structure), epa, fmax\n\n\ndef relax_fixed_cell(optimizer, structure, fmax, steps):\n    \"\"\"Version-aware fixed-cell relaxation. Raise real optimizer errors rather than hiding them.\"\"\"\n    parameters = inspect.signature(optimizer.relax).parameters\n    accepts_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in parameters.values())\n    kwargs = {\"fmax\": fmax, \"steps\": steps}\n    if \"relax_cell\" in parameters or accepts_kwargs:\n        kwargs[\"relax_cell\"] = False\n    if \"verbose\" in parameters or accepts_kwargs:\n        kwargs[\"verbose\"] = False\n    captured = StringIO()\n    with redirect_stdout(captured), redirect_stderr(captured):\n        result = optimizer.relax(structure.copy(), **kwargs)\n    if not isinstance(result, dict) or result.get(\"final_structure\") is None:\n        raise RuntimeError(\"CHGNet optimizer did not return a final_structure.\")\n    final_structure = result[\"final_structure\"]\n    if not isinstance(final_structure, Structure) or len(final_structure) != len(structure):\n        raise RuntimeError(\"CHGNet returned an unexpected final structure or changed atom count.\")\n    return final_structure\n\n\ndef fmt_hkl(hkl):\n    return \"(\" + \",\".join(str(int(x)) for x in hkl) + \")\"\n\n\ndef file_hkl(hkl):\n    return \"_\".join((f\"m{abs(x)}\" if x < 0 else str(x)) for x in hkl)\n\n\ndef surface_fingerprint(slab, side, depth_A, distance_tol_A):\n    \"\"\"Translation/in-plane-periodic fingerprint of atoms close to one slab face.\n\n    The top and bottom fingerprint is formed from atoms within a user-controlled depth.\n    It uses species, distance-from-face and periodic pair distances; no atom identity labels\n    or hand-picked MOF-specific indices are used.\n    \"\"\"\n    matrix = np.asarray(slab.lattice.matrix, dtype=float)\n    a_vec, b_vec, c_vec = matrix\n    normal = np.cross(a_vec, b_vec)\n    norm = float(np.linalg.norm(normal))\n    if norm <= 0:\n        raise ValueError(\"Cannot determine slab normal for surface fingerprint.\")\n    normal /= norm\n    if np.dot(normal, c_vec) < 0:\n        normal *= -1\n    coords = np.asarray(slab.cart_coords, dtype=float)\n    heights = coords @ normal\n    edge = float(np.max(heights) if side == \"top\" else np.min(heights))\n    depth = (edge - heights) if side == \"top\" else (heights - edge)\n    indices = np.where(depth <= depth_A + 1e-8)[0]\n    if len(indices) == 0:\n        indices = np.array([int(np.argmax(heights) if side == \"top\" else np.argmin(heights))])\n    symbols = [slab[i].specie.symbol for i in indices]\n    frac = np.asarray(slab.frac_coords, dtype=float)\n\n    atom_signature = tuple(sorted(\n        (symbols[q], int(round(float(depth[i]) / distance_tol_A)))\n        for q, i in enumerate(indices)\n    ))\n    pair_counts = Counter()\n    inplane_shifts = list(itertools.product((-1, 0, 1), repeat=2))\n    for q in range(len(indices)):\n        i = int(indices[q])\n        for r in range(q + 1, len(indices)):\n            j = int(indices[r])\n            delta_xy = frac[i, :2] - frac[j, :2]\n            best_parallel = np.inf\n            for shift in inplane_shifts:\n                vec_xy = (delta_xy + np.asarray(shift, dtype=float)) @ matrix[:2]\n                dxy = float(np.linalg.norm(vec_xy))\n                if dxy < best_parallel:\n                    best_parallel = dxy\n            dz = float(depth[i] - depth[j])\n            dist = math.sqrt(best_parallel * best_parallel + dz * dz)\n            sp_pair = tuple(sorted((slab[i].specie.symbol, slab[j].specie.symbol)))\n            quantized = int(round(dist / distance_tol_A))\n            pair_counts[(sp_pair, quantized)] += 1\n    canonical = (atom_signature, tuple(sorted(pair_counts.items(), key=repr)))\n    return hashlib.sha1(repr(canonical).encode(\"utf-8\")).hexdigest()[:20]\n\n# -----------------------------------------------------------------------------\n# 4. Relax the bulk reference at fixed cell.\n# -----------------------------------------------------------------------------\nmodel = None\noptimizer = None\nbulk_final = working_structure.copy()\nbulk_total_energy_eV = None\nbulk_energy_per_atom_eV = None\nbulk_fmax_eV_A = None\nbulk_ok = False\nbulk_error = None\n\ntry:\n    model = CHGNet.load()\n    optimizer = StructOptimizer(model=model)\n    print(\"CHGNet loaded. Relaxing the bulk at fixed cell...\")\n    bulk_final = relax_fixed_cell(optimizer, working_structure, FORCE_TOL_EV_A, MAX_RELAX_STEPS)\n    bulk_total_energy_eV, bulk_energy_per_atom_eV, bulk_fmax_eV_A = model_energy_force(model, bulk_final)\n    bulk_ok = bool(bulk_fmax_eV_A <= FORCE_TOL_EV_A)\n    bulk_final.to(filename=str(RESULTS_DIR / f\"{safe_name}_bulk_relaxed_fixed_cell.cif\"))\n    print(f\"Bulk total energy : {bulk_total_energy_eV:.8f} eV\")\n    print(f\"Bulk energy/atom  : {bulk_energy_per_atom_eV:.8f} eV/atom\")\n    print(f\"Bulk max force    : {bulk_fmax_eV_A:.6f} eV/A\")\n    print(f\"Bulk converged    : {bulk_ok}\")\n    if not bulk_ok:\n        bulk_error = \"Bulk max force exceeded the selected tolerance; surface energies are disabled.\"\nexcept Exception as exc:\n    bulk_error = f\"{type(exc).__name__}: {exc}\"\n    print(\"FATAL FOR MLIP SURFACE ENERGIES: bulk relaxation/reference failed:\", bulk_error)\n    model = None\n    optimizer = None\n    bulk_ok = False\n\n# -----------------------------------------------------------------------------\n# 5. Generate directed (signed) Miller orientation orbits.\n# -----------------------------------------------------------------------------\nmiller_reps, miller_orbits, direction_to_rep, all_directed_hkls, miller_notes = directed_miller_orbits(\n    working_structure, MAX_MILLER_INDEX, SYMPREC_A\n)\nprint(f\"\\nRequested signed Miller range: |h|,|k|,|l| <= {MAX_MILLER_INDEX}\")\nprint(f\"Directed primitive Miller orientations: {len(all_directed_hkls)}\")\nprint(f\"Symmetry-reduced directed surface classes: {len(miller_reps)}\")\nfor note in miller_notes:\n    print(\"Miller note:\", note)\nprint(\"No signed orientation is merged with its opposite unless a real point-group operation maps them.\")\n\n# -----------------------------------------------------------------------------\n# 6. Generate stoichiometric slabs; relax asymmetric slabs too. Energies are pair sums.\n# -----------------------------------------------------------------------------\nrecords = []\nunique_relaxed_records = []\nprimary_pair_files = {}\n\nfor facet_i, hkl in enumerate(miller_reps, start=1):\n    print(f\"\\n[{facet_i}/{len(miller_reps)}] Directed facet class representative {fmt_hkl(hkl)}\")\n    generation_errors = []\n    generated = None\n    generator_primitive_mode = None\n    for use_primitive_surface_cell in (True, False):\n        try:\n            generator = SlabGenerator(\n                initial_structure=bulk_final,\n                miller_index=hkl,\n                min_slab_size=SLAB_THICKNESS_A,\n                min_vacuum_size=VACUUM_A,\n                in_unit_planes=False,\n                center_slab=True,\n                primitive=use_primitive_surface_cell,\n                reorient_lattice=True,\n            )\n            get_params = inspect.signature(generator.get_slabs).parameters\n            get_kwargs = {}\n            if \"symmetrize\" in get_params:\n                get_kwargs[\"symmetrize\"] = False  # asymmetric faces are handled by pair-energy equations\n            if \"filter_out_sym_slabs\" in get_params:\n                get_kwargs[\"filter_out_sym_slabs\"] = False  # retain all unique termination candidates\n            if \"ftol\" in get_params:\n                get_kwargs[\"ftol\"] = TERMINATION_SHIFT_TOL_FRACTIONAL\n            # SYMMETRIC-SLAB CHANGE: keep linkers whole if pymatgen supports it and at least one\n            # such termination exists; otherwise fall back to unrestricted cutting and say so.\n            if KEEP_LINKERS_INTACT and \"bonds\" in get_params and \"max_broken_bonds\" in get_params:\n                present = {str(e) for e in bulk_final.composition.elements}\n                bonds = {k: v for k, v in LINKER_BONDS.items() if set(k) <= present}\n                try:\n                    trial = generator.get_slabs(**dict(get_kwargs, bonds=bonds, max_broken_bonds=0))\n                except Exception as bond_exc:\n                    trial = []\n                    print(f\"  Linker-preserving cut failed: {type(bond_exc).__name__}: {bond_exc}\")\n                if trial:\n                    get_kwargs[\"bonds\"] = bonds\n                    get_kwargs[\"max_broken_bonds\"] = 0\n                    print(f\"  Linker-preserving terminations: {len(trial)} (C-C, C-H, C-O, C-N bonds kept whole)\")\n                else:\n                    print(\"  No termination keeps every linker whole; unrestricted cutting is used for this facet.\")\n            generated = list(generator.get_slabs(**get_kwargs))\n            slab_modes = [\"as_cut\"] * len(generated)\n            # SYMMETRIC-SLAB CHANGE: also ask pymatgen for symmetrized slabs, whose two faces\n            # carry the same termination, so that gamma = pair excess / 2 for that termination.\n            if SYMMETRIC_SLABS and \"symmetrize\" in get_params:\n                try:\n                    sym_kwargs = dict(get_kwargs)\n                    sym_kwargs[\"symmetrize\"] = True\n                    sym_generated = list(generator.get_slabs(**sym_kwargs))\n                    generated += sym_generated\n                    slab_modes += [\"symmetrized\"] * len(sym_generated)\n                    print(f\"  Symmetrized termination candidates added: {len(sym_generated)}\")\n                except Exception as sym_exc:\n                    print(f\"  Symmetrized slab generation failed: {type(sym_exc).__name__}: {sym_exc}\")\n            generator_primitive_mode = bool(use_primitive_surface_cell)\n            break\n        except Exception as exc:\n            generation_errors.append(\n                f\"primitive={use_primitive_surface_cell}: {type(exc).__name__}: {exc}\"\n            )\n            generated = None\n\n    if generated is None:\n        exc_detail = \" | \".join(generation_errors)\n        records.append({\n            \"h\": hkl[0], \"k\": hkl[1], \"l\": hkl[2], \"miller\": fmt_hkl(hkl),\n            \"termination\": 0, \"n_atoms\": 0, \"area_A2\": np.nan,\n            \"stoichiometric\": False, \"symmetric_diagnostic\": False,\n            \"top_fingerprint\": \"\", \"bottom_fingerprint\": \"\",\n            \"slab_total_energy_eV\": np.nan, \"pair_excess_eV_A2\": np.nan,\n            \"fit_rank\": 0, \"fit_unknowns\": 0, \"fit_rms_eV_A2\": np.nan,\n            \"status\": \"slab_generation_failed\", \"detail\": exc_detail,\n            \"initial_slab_file\": \"\", \"relaxed_slab_file\": \"\",\n            \"slab_generator_primitive_surface_cell\": None,\n        })\n        print(f\"  Slab generation failed in both surface-cell modes: {exc_detail}\")\n        continue\n\n    print(f\"  Slab surface-cell mode: primitive={generator_primitive_mode}\")\n    print(f\"  Generated raw termination candidates: {len(generated)}\")\n    seen_pairs = set()\n    retained_this_facet = 0\n    for term_i, slab in enumerate(generated, start=1):\n        base = {\n            \"h\": hkl[0], \"k\": hkl[1], \"l\": hkl[2], \"miller\": fmt_hkl(hkl),\n            \"termination\": int(term_i), \"n_atoms\": int(len(slab)),\n            \"area_A2\": np.nan, \"stoichiometric\": False,\n            \"symmetric_diagnostic\": False, \"symmetric_after_relaxation\": False,\n            \"top_fingerprint\": \"\", \"bottom_fingerprint\": \"\",\n            \"top_fingerprint_relaxed\": \"\", \"bottom_fingerprint_relaxed\": \"\",\n            \"slab_total_energy_eV\": np.nan, \"pair_excess_eV_A2\": np.nan,\n            \"fit_rank\": 0, \"fit_unknowns\": 0, \"fit_rms_eV_A2\": np.nan,\n            \"status\": \"\", \"detail\": \"\", \"initial_slab_file\": \"\", \"relaxed_slab_file\": \"\",\n            \"slab_generator_primitive_surface_cell\": generator_primitive_mode,\n            \"slab_mode\": slab_modes[term_i - 1],  # SYMMETRIC-SLAB CHANGE\n        }\n        try:\n            area = surface_area_A2(slab)\n            base[\"area_A2\"] = area\n            base[\"stoichiometric\"] = bool(composition_is_proportional(slab, working_structure))\n            try:\n                base[\"symmetric_diagnostic\"] = bool(slab.is_symmetric(symprec=SYMPREC_A))\n            except Exception:\n                base[\"symmetric_diagnostic\"] = False\n            if not base[\"stoichiometric\"]:\n                base[\"status\"] = \"excluded_nonstoichiometric\"\n                base[\"detail\"] = \"Non-stoichiometric slab: chemical-potential reference terms are required; excluded from this fit.\"\n                records.append(base)\n                continue\n            top_fp = surface_fingerprint(slab, \"top\", SURFACE_FINGERPRINT_DEPTH_A, FINGERPRINT_DISTANCE_TOL_A)\n            bottom_fp = surface_fingerprint(slab, \"bottom\", SURFACE_FINGERPRINT_DEPTH_A, FINGERPRINT_DISTANCE_TOL_A)\n            base[\"top_fingerprint\"] = top_fp\n            base[\"bottom_fingerprint\"] = bottom_fp\n            pair_key = (int(len(slab)), round(area, 4), top_fp, bottom_fp)\n            if pair_key in seen_pairs:\n                base[\"status\"] = \"duplicate_surface_pair_not_relaxed\"\n                base[\"detail\"] = \"Same atom count, area, and ordered top/bottom surface fingerprints as an earlier candidate; skipped to avoid repeating the same slab relaxation.\"\n                base[\"initial_slab_file\"] = primary_pair_files.get((hkl, pair_key), \"\")\n                records.append(base)\n                continue\n            seen_pairs.add(pair_key)\n            retained_this_facet += 1\n            initial_path = INITIAL_SLAB_DIR / f\"{safe_name}_hkl_{file_hkl(hkl)}_term_{term_i:04d}.cif\"\n            slab.to(filename=str(initial_path))\n            base[\"initial_slab_file\"] = str(initial_path)\n            primary_pair_files[(hkl, pair_key)] = str(initial_path)\n\n            if model is None or optimizer is None or not bulk_ok or bulk_energy_per_atom_eV is None:\n                base[\"status\"] = \"surface_energy_disabled\"\n                base[\"detail\"] = bulk_error or \"A converged bulk reference is unavailable.\"\n                records.append(base)\n                continue\n\n            relaxed = relax_fixed_cell(optimizer, slab, FORCE_TOL_EV_A, MAX_RELAX_STEPS)\n            slab_total_eV, _slab_epa, slab_fmax = model_energy_force(model, relaxed)\n            base[\"fmax_eV_A\"] = float(slab_fmax)\n            if slab_fmax > FORCE_TOL_EV_A:\n                base[\"status\"] = \"excluded_not_converged\"\n                base[\"detail\"] = f\"Relaxed slab max force {slab_fmax:.6f} eV/A exceeds tolerance {FORCE_TOL_EV_A:.6f} eV/A.\"\n                records.append(base)\n                continue\n            if not composition_is_proportional(relaxed, working_structure):\n                base[\"status\"] = \"excluded_relaxed_composition_mismatch\"\n                base[\"detail\"] = \"The relaxed slab composition no longer matches the bulk formula ratio.\"\n                records.append(base)\n                continue\n\n            # Fingerprint the relaxed face terminations, because the final energy belongs to\n            # the relaxed surface state. Large reconstructions become distinct fingerprint IDs.\n            base[\"top_fingerprint_relaxed\"] = surface_fingerprint(\n                relaxed, \"top\", SURFACE_FINGERPRINT_DEPTH_A, FINGERPRINT_DISTANCE_TOL_A\n            )\n            base[\"bottom_fingerprint_relaxed\"] = surface_fingerprint(\n                relaxed, \"bottom\", SURFACE_FINGERPRINT_DEPTH_A, FINGERPRINT_DISTANCE_TOL_A\n            )\n            try:\n                base[\"symmetric_after_relaxation\"] = bool(relaxed.is_symmetric(symprec=SYMPREC_A))\n            except Exception:\n                base[\"symmetric_after_relaxation\"] = False\n            # If pymatgen verifies the relaxed slab's face-exchanging symmetry, the two\n            # faces are the same surface state. Use one shared unknown in the pair equations\n            # even if numerical fingerprinting separated them by a small geometric detail.\n            # SYMMETRIC-SLAB CHANGE: a slab built symmetric (verified before relaxation) also\n            # keeps one shared unknown; relaxation is unconstrained and can break the symmetry\n            # by small amounts. The two cases are recorded separately in the CSV.\n            if base[\"symmetric_after_relaxation\"] or base[\"symmetric_diagnostic\"]:\n                base[\"bottom_fingerprint_relaxed\"] = base[\"top_fingerprint_relaxed\"]\n            # For possibly inequivalent faces this is gamma_top + gamma_bottom, NOT one gamma.\n            pair_excess = (slab_total_eV - len(relaxed) * bulk_energy_per_atom_eV) / area\n            relaxed_path = RELAXED_SLAB_DIR / f\"{safe_name}_hkl_{file_hkl(hkl)}_term_{term_i:04d}.cif\"\n            relaxed.to(filename=str(relaxed_path))\n            base[\"slab_total_energy_eV\"] = float(slab_total_eV)\n            base[\"pair_excess_eV_A2\"] = float(pair_excess)\n            base[\"fmax_eV_A\"] = float(slab_fmax)\n            base[\"relaxed_slab_file\"] = str(relaxed_path)\n            base[\"status\"] = \"pair_energy_available\"\n            base[\"detail\"] = \"Stoichiometric, force-converged slab. Stored value is the sum of both surface energies per area; individual gammas are solved below.\"\n            records.append(base)\n            unique_relaxed_records.append(base.copy())\n            print(f\"  Termination {term_i}: relaxed; N={len(relaxed)}; area={area:.3f} A^2; pair excess={pair_excess:.5f} eV/A^2; fmax={slab_fmax:.4f} eV/A\")\n        except Exception as exc:\n            base[\"status\"] = \"slab_processing_failed\"\n            base[\"detail\"] = f\"{type(exc).__name__}: {exc}\"\n            records.append(base)\n            print(f\"  Termination {term_i} failed: {type(exc).__name__}: {exc}\")\n    print(f\"  Unique stoichiometric surface pairs kept for relaxation: {retained_this_facet}\")\n\n# -----------------------------------------------------------------------------\n# 7. Deconvolve pair energies into termination-specific surface energies.\n# Each row satisfies excess/A = gamma(top fingerprint) + gamma(bottom fingerprint).\n# Non-stoichiometric records are never included in this simple-reference fit.\n# -----------------------------------------------------------------------------\nrecords_df = pd.DataFrame(records)\nif records_df.empty:\n    records_df = pd.DataFrame(columns=[\n        \"h\", \"k\", \"l\", \"miller\", \"termination\", \"n_atoms\", \"area_A2\", \"stoichiometric\",\n        \"symmetric_diagnostic\", \"symmetric_after_relaxation\", \"top_fingerprint\", \"bottom_fingerprint\",\n        \"top_fingerprint_relaxed\", \"bottom_fingerprint_relaxed\", \"slab_total_energy_eV\",\n        \"pair_excess_eV_A2\", \"fmax_eV_A\", \"fit_rank\", \"fit_unknowns\", \"fit_rms_eV_A2\",\n        \"status\", \"detail\", \"initial_slab_file\", \"relaxed_slab_file\"\n    ])\n\nsurface_gamma_rows = []\nfit_summary_rows = []\nvalid_pair_df = records_df[records_df.get(\"status\", pd.Series(dtype=str)) == \"pair_energy_available\"].copy()\n\nfor hkl in miller_reps:\n    group = valid_pair_df[\n        (pd.to_numeric(valid_pair_df[\"h\"], errors=\"coerce\") == hkl[0]) &\n        (pd.to_numeric(valid_pair_df[\"k\"], errors=\"coerce\") == hkl[1]) &\n        (pd.to_numeric(valid_pair_df[\"l\"], errors=\"coerce\") == hkl[2])\n    ].copy()\n    fit_row = {\n        \"h\": hkl[0], \"k\": hkl[1], \"l\": hkl[2], \"miller\": fmt_hkl(hkl),\n        \"n_pair_equations\": int(len(group)), \"n_surface_terminations\": 0,\n        \"matrix_rank\": 0, \"matrix_columns\": 0, \"fit_rms_eV_A2\": np.nan,\n        \"fit_status\": \"no_valid_pair_energies\", \"detail\": \"No stoichiometric, force-converged slab pairs were available.\"\n    }\n    if group.empty:\n        fit_summary_rows.append(fit_row)\n        continue\n\n    terminations = sorted(set(group[\"top_fingerprint_relaxed\"].astype(str)) | set(group[\"bottom_fingerprint_relaxed\"].astype(str)))\n    col_index = {fp: i for i, fp in enumerate(terminations)}\n    A = np.zeros((len(group), len(terminations)), dtype=float)\n    y = pd.to_numeric(group[\"pair_excess_eV_A2\"], errors=\"coerce\").to_numpy(dtype=float)\n    for row_i, row in enumerate(group.itertuples()):\n        A[row_i, col_index[str(row.top_fingerprint_relaxed)]] += 1.0\n        A[row_i, col_index[str(row.bottom_fingerprint_relaxed)]] += 1.0\n    rank = int(np.linalg.matrix_rank(A))\n    n_equations = int(len(group))\n    n_unknowns = int(len(terminations))\n    n_redundant_equations = max(0, n_equations - n_unknowns)\n    fit_row.update({\n        \"n_surface_terminations\": n_unknowns,\n        \"matrix_rank\": rank,\n        \"matrix_columns\": n_unknowns,\n        \"n_redundant_equations\": n_redundant_equations,\n        \"has_redundant_equations\": bool(n_redundant_equations > 0),\n        \"fit_validation\": \"pending\",\n    })\n    # SYMMETRIC-SLAB CHANGE: an underdetermined system can still fix some unknowns exactly\n    # (for example a termination seen on both faces of a symmetric slab). Column j is\n    # identifiable when every null-space vector of A has a zero j-th component.\n    if rank < len(terminations):\n        _u, _s, _vt = np.linalg.svd(A)\n        null_basis = _vt[rank:]\n        identifiable = [fp for fp, j in col_index.items() if np.all(np.abs(null_basis[:, j]) < 1e-8)]\n        if identifiable:\n            solution, _resid, _rank, _singular = np.linalg.lstsq(A, y, rcond=None)\n            residuals = A @ solution - y\n            rms = float(np.sqrt(np.mean(residuals ** 2)))\n            fit_row[\"fit_rms_eV_A2\"] = rms\n            fit_row[\"n_identifiable_terminations\"] = len(identifiable)\n            if np.isfinite(rms) and rms <= PAIR_FIT_RMS_TOL_EV_A2:\n                fit_row[\"fit_status\"] = \"fit_partial_identified\"\n                fit_row[\"fit_validation\"] = \"partial_identifiable_subset\"\n                fit_row[\"detail\"] = (\n                    f\"Rank {rank} < {len(terminations)} unknowns, but {len(identifiable)} termination energies are \"\n                    \"uniquely fixed (null-space test). Only those are reported; the lowest of them is an upper bound \"\n                    \"on the facet's lowest termination energy.\"\n                )\n                for fp in identifiable:\n                    gamma_eV_A2 = float(solution[col_index[fp]])\n                    positive = bool(np.isfinite(gamma_eV_A2) and gamma_eV_A2 > 0)\n                    surface_gamma_rows.append({\n                        \"h\": hkl[0], \"k\": hkl[1], \"l\": hkl[2], \"miller\": fmt_hkl(hkl),\n                        \"termination_fingerprint\": fp, \"gamma_eV_A2\": gamma_eV_A2,\n                        \"gamma_J_m2\": gamma_eV_A2 * 16.02176634,\n                        \"positive_gamma\": positive, \"fit_rank\": rank, \"fit_unknowns\": len(terminations),\n                        \"fit_rms_eV_A2\": rms,\n                        \"fit_status\": \"fit_partial_identified\" if positive else \"fit_partial_nonpositive_gamma\",\n                        \"fit_validation\": \"partial_identifiable_subset\",\n                        \"n_pair_equations\": n_equations,\n                        \"n_redundant_equations\": n_redundant_equations,\n                    })\n                fit_summary_rows.append(fit_row)\n                continue\n    if rank < len(terminations):\n        fit_row[\"fit_status\"] = \"underdetermined_surface_energies\"\n        fit_row[\"detail\"] = (\n            f\"Pair-equation matrix rank {rank} is smaller than {len(terminations)} unknown termination energies. \"\n            \"Individual surface energies cannot be uniquely separated from these slabs.\"\n        )\n        records_df.loc[(records_df[\"h\"] == hkl[0]) & (records_df[\"k\"] == hkl[1]) & (records_df[\"l\"] == hkl[2]) & (records_df[\"status\"] == \"pair_energy_available\"), \"status\"] = \"pair_fit_underdetermined\"\n        records_df.loc[(records_df[\"h\"] == hkl[0]) & (records_df[\"k\"] == hkl[1]) & (records_df[\"l\"] == hkl[2]) & (records_df[\"status\"] == \"pair_fit_underdetermined\"), \"detail\"] = fit_row[\"detail\"]\n        fit_summary_rows.append(fit_row)\n        continue\n\n    solution, _resid, _rank, _singular = np.linalg.lstsq(A, y, rcond=None)\n    residuals = A @ solution - y\n    rms = float(np.sqrt(np.mean(residuals ** 2)))\n    fit_row[\"fit_rms_eV_A2\"] = rms\n    if not np.isfinite(rms) or rms > PAIR_FIT_RMS_TOL_EV_A2:\n        fit_row[\"fit_status\"] = \"fit_residual_exceeds_tolerance\"\n        fit_row[\"detail\"] = (\n            f\"Pair-energy RMS residual {rms:.6f} eV/A^2 exceeds user tolerance \"\n            f\"{PAIR_FIT_RMS_TOL_EV_A2:.6f} eV/A^2. Increase slab thickness or review surface fingerprints/model consistency.\"\n        )\n        fit_summary_rows.append(fit_row)\n        records_df.loc[(records_df[\"h\"] == hkl[0]) & (records_df[\"k\"] == hkl[1]) & (records_df[\"l\"] == hkl[2]) & (records_df[\"status\"] == \"pair_energy_available\"), \"status\"] = \"pair_fit_residual_too_high\"\n        records_df.loc[(records_df[\"h\"] == hkl[0]) & (records_df[\"k\"] == hkl[1]) & (records_df[\"l\"] == hkl[2]) & (records_df[\"status\"] == \"pair_fit_residual_too_high\"), \"detail\"] = fit_row[\"detail\"]\n        continue\n\n    fit_row[\"fit_status\"] = \"fit_ok\"\n    if n_redundant_equations > 0:\n        fit_row[\"fit_validation\"] = \"overdetermined_residual_checked\"\n        fit_row[\"detail\"] = (\n            \"Full-rank pair-equation solution with redundant equations; RMS residual is within the user-specified tolerance. \"\n            \"This is an internal consistency check, not a substitute for slab-thickness/vacuum convergence or DFT validation.\"\n        )\n    else:\n        fit_row[\"fit_validation\"] = \"full_rank_no_redundancy\"\n        fit_row[\"detail\"] = (\n            \"The pair-equation matrix is full rank, so the termination energies are algebraically identifiable, \"\n            \"but there are no redundant equations to independently test pair-model consistency. Treat as provisional \"\n            \"until slab-thickness/vacuum convergence and independent validation are completed.\"\n        )\n    gamma_map = {fp: float(solution[i]) for fp, i in col_index.items()}\n    for fp, gamma_eV_A2 in gamma_map.items():\n        positive = bool(np.isfinite(gamma_eV_A2) and gamma_eV_A2 > 0)\n        surface_gamma_rows.append({\n            \"h\": hkl[0], \"k\": hkl[1], \"l\": hkl[2], \"miller\": fmt_hkl(hkl),\n            \"termination_fingerprint\": fp, \"gamma_eV_A2\": gamma_eV_A2,\n            \"gamma_J_m2\": gamma_eV_A2 * 16.02176634,\n            \"positive_gamma\": positive, \"fit_rank\": rank, \"fit_unknowns\": len(terminations),\n            \"fit_rms_eV_A2\": rms,\n            \"fit_status\": \"fit_ok\" if positive else \"fit_ok_nonpositive_gamma\",\n            \"fit_validation\": fit_row[\"fit_validation\"],\n            \"n_pair_equations\": n_equations,\n            \"n_redundant_equations\": n_redundant_equations,\n        })\n    records_df.loc[(records_df[\"h\"] == hkl[0]) & (records_df[\"k\"] == hkl[1]) & (records_df[\"l\"] == hkl[2]) & (records_df[\"status\"] == \"pair_energy_available\"), \"fit_rank\"] = rank\n    records_df.loc[(records_df[\"h\"] == hkl[0]) & (records_df[\"k\"] == hkl[1]) & (records_df[\"l\"] == hkl[2]) & (records_df[\"status\"] == \"pair_energy_available\"), \"fit_unknowns\"] = len(terminations)\n    records_df.loc[(records_df[\"h\"] == hkl[0]) & (records_df[\"k\"] == hkl[1]) & (records_df[\"l\"] == hkl[2]) & (records_df[\"status\"] == \"pair_energy_available\"), \"fit_rms_eV_A2\"] = rms\n    fit_summary_rows.append(fit_row)\n\nfit_summary_df = pd.DataFrame(fit_summary_rows).drop_duplicates(subset=[\"h\", \"k\", \"l\"], keep=\"last\")\nsurface_gamma_df = pd.DataFrame(surface_gamma_rows)\nfit_csv = RESULTS_DIR / f\"{safe_name}_surface_pair_fit_summary.csv\"\nsurface_gamma_csv = RESULTS_DIR / f\"{safe_name}_surface_energies_individual.csv\"\nrecords_csv = RESULTS_DIR / f\"{safe_name}_surface_pair_records.csv\"\nrecords_df.to_csv(records_csv, index=False)\nfit_summary_df.to_csv(fit_csv, index=False)\nsurface_gamma_df.to_csv(surface_gamma_csv, index=False)\n\nprint(\"\\n================ SURFACE PAIR FIT SUMMARY ================\")\nif len(fit_summary_df):\n    display(fit_summary_df)\nelse:\n    print(\"No facet-fit diagnostics were produced.\")\nprint(\"\\nIndividual termination surface energies:\")\nif len(surface_gamma_df):\n    display(surface_gamma_df)\nelse:\n    print(\"No individual surface energies were identified from the pair equations.\")\nprint(f\"Pair records CSV : {records_csv}\")\nprint(f\"Fit summary CSV  : {fit_csv}\")\nprint(f\"Surface gamma CSV: {surface_gamma_csv}\")\n\n# -----------------------------------------------------------------------------\n# 8. Convert solved termination energies into signed-direction energy scenarios.\n# The minimum is the equilibrium termination choice; median/maximum are sensitivity\n# scenarios only, not separate equilibrium predictions.\n# -----------------------------------------------------------------------------\nfit_lookup = {}\nfor row in surface_gamma_df.itertuples():\n    fit_lookup.setdefault((int(row.h), int(row.k), int(row.l)), {})[str(row.termination_fingerprint)] = float(row.gamma_J_m2)\n\nscenario_names = (\"minimum\", \"median\", \"maximum\")\nraw_direction_scenarios = {name: defaultdict(list) for name in scenario_names}\nraw_direction_invalid = defaultdict(list)\nfor rep in miller_reps:\n    rep_group = valid_pair_df[\n        (pd.to_numeric(valid_pair_df[\"h\"], errors=\"coerce\") == rep[0]) &\n        (pd.to_numeric(valid_pair_df[\"k\"], errors=\"coerce\") == rep[1]) &\n        (pd.to_numeric(valid_pair_df[\"l\"], errors=\"coerce\") == rep[2])\n    ]\n    fit_state = fit_summary_df[\n        (fit_summary_df[\"h\"] == rep[0]) & (fit_summary_df[\"k\"] == rep[1]) & (fit_summary_df[\"l\"] == rep[2])\n    ]\n    # SYMMETRIC-SLAB CHANGE: partially identified facets are used as well.\n    if fit_state.empty or fit_state.iloc[-1][\"fit_status\"] not in (\"fit_ok\", \"fit_partial_identified\"):\n        continue\n    partial_fit = fit_state.iloc[-1][\"fit_status\"] == \"fit_partial_identified\"\n    gamma_by_fp = fit_lookup.get(rep, {})\n    top_fps = sorted(set(rep_group[\"top_fingerprint_relaxed\"].astype(str)))\n    bottom_fps = sorted(set(rep_group[\"bottom_fingerprint_relaxed\"].astype(str)))\n\n    def face_gamma_scenarios(fps, direction):\n        if not fps:\n            return None\n        vals = [gamma_by_fp[fp] for fp in fps if fp in gamma_by_fp]\n        if not vals or (len(vals) != len(fps) and not partial_fit):  # SYMMETRIC-SLAB CHANGE\n            return None\n        if any(not np.isfinite(v) or v <= 0 for v in vals):\n            raw_direction_invalid[direction].append(\n                f\"At least one fitted termination energy for {fmt_hkl(direction)} is non-positive or non-finite.\"\n            )\n            return None\n        vals = np.asarray(vals, dtype=float)\n        return {\n            \"minimum\": float(np.min(vals)),\n            \"median\": float(np.median(vals)),\n            \"maximum\": float(np.max(vals)),\n        }\n\n    top_scenarios = face_gamma_scenarios(top_fps, rep)\n    opposite = tuple(-v for v in rep)\n    bottom_scenarios = face_gamma_scenarios(bottom_fps, opposite)\n    if top_scenarios is not None:\n        for direction in miller_orbits[rep]:\n            for scenario in scenario_names:\n                raw_direction_scenarios[scenario][direction].append(top_scenarios[scenario])\n    if bottom_scenarios is not None:\n        opposite_rep = direction_to_rep.get(opposite)\n        if opposite_rep is not None:\n            for direction in miller_orbits[opposite_rep]:\n                for scenario in scenario_names:\n                    raw_direction_scenarios[scenario][direction].append(bottom_scenarios[scenario])\n\n# Multiple symmetry-equivalent paths may assign a direction more than once.\n# Combine those assignments conservatively and retain diagnostics when they disagree.\ndirection_gamma_scenarios = {name: {} for name in scenario_names}\ndirection_gamma_notes = []\nfor scenario in scenario_names:\n    for direction, vals in raw_direction_scenarios[scenario].items():\n        vals = [float(v) for v in vals if np.isfinite(v) and v > 0]\n        if not vals:\n            continue\n        if len(vals) > 1 and max(vals) - min(vals) > PAIR_FIT_RMS_TOL_EV_A2 * 16.02176634:\n            direction_gamma_notes.append(\n                f\"Symmetry-equivalent estimates differ for {fmt_hkl(direction)} in the {scenario} scenario; combined by {scenario}.\"\n            )\n        if scenario == \"minimum\":\n            combined = min(vals)\n        elif scenario == \"median\":\n            combined = float(np.median(vals))\n        else:\n            combined = max(vals)\n        direction_gamma_scenarios[scenario][direction] = combined\n\ndirection_gamma_J_m2 = direction_gamma_scenarios[\"minimum\"]\n\n# SYMMETRIC-SLAB CHANGE: record where each directed energy came from, and fill facet classes\n# that still have none with half of their lowest pair excess. For any slab,\n# min(gamma_top, gamma_bottom) <= (gamma_top + gamma_bottom)/2, so this is an UPPER BOUND on\n# the lowest termination energy of that facet, not a fitted value. It assigns the same number\n# to (hkl) and (-h-k-l); in a non-centrosymmetric crystal these can differ.\ndirection_gamma_source = {}\nfor rep in miller_reps:\n    fs = fit_summary_df[(fit_summary_df[\"h\"] == rep[0]) & (fit_summary_df[\"k\"] == rep[1]) & (fit_summary_df[\"l\"] == rep[2])]\n    status = fs.iloc[-1][\"fit_status\"] if len(fs) else \"none\"\n    tag = {\"fit_ok\": \"fitted\", \"fit_partial_identified\": \"partial_fit_upper_bound\"}.get(status)\n    if tag:\n        for direction in miller_orbits[rep]:\n            if direction in direction_gamma_J_m2:\n                direction_gamma_source[direction] = tag\nif PAIR_AVERAGE_FALLBACK:\n    for rep in miller_reps:\n        rep_group = valid_pair_df[\n            (pd.to_numeric(valid_pair_df[\"h\"], errors=\"coerce\") == rep[0]) &\n            (pd.to_numeric(valid_pair_df[\"k\"], errors=\"coerce\") == rep[1]) &\n            (pd.to_numeric(valid_pair_df[\"l\"], errors=\"coerce\") == rep[2])\n        ]\n        excess = pd.to_numeric(rep_group[\"pair_excess_eV_A2\"], errors=\"coerce\").dropna()\n        excess = excess[excess > 0]\n        if excess.empty:\n            continue\n        bound = float(excess.min()) / 2.0 * 16.02176634\n        opposite_rep = direction_to_rep.get(tuple(-v for v in rep))\n        for orbit_rep in (rep, opposite_rep):\n            if orbit_rep is None:\n                continue\n            for direction in miller_orbits[orbit_rep]:\n                if direction_gamma_source.get(direction) == \"fitted\":\n                    continue\n                old = direction_gamma_J_m2.get(direction)\n                if old is None or bound < old:\n                    direction_gamma_J_m2[direction] = bound\n                    direction_gamma_source[direction] = \"pair_average_upper_bound\"\nestimated_directions = sorted(d for d, s in direction_gamma_source.items() if s != \"fitted\")\nif estimated_directions:\n    print(f\"{len(estimated_directions)} directed energies are upper-bound estimates \"\n          \"(partial fit or pair average), not fitted termination energies.\")\n\n# -----------------------------------------------------------------------------\n# 9. Signed-normal Wulff and explicitly labelled geometry-only controls.\n# -----------------------------------------------------------------------------\ndef direction_normal(lattice, hkl):\n    rec = np.asarray(lattice.reciprocal_lattice_crystallographic.matrix, dtype=float)\n    normal = np.asarray(hkl, dtype=float) @ rec\n    norm = float(np.linalg.norm(normal))\n    if norm <= 0 or not np.isfinite(norm):\n        raise ValueError(f\"Invalid reciprocal normal for {hkl}.\")\n    return normal / norm\n\n\ndef build_polyhedron(label, directed_gamma, lattice):\n    \"\"\"Build an oriented Wulff polyhedron from half-spaces n.r <= gamma.\n\n    Energies need only be relative: all distances are divided by the minimum positive gamma.\n    This custom construction supports distinct energies for +hkl and -hkl.\n    \"\"\"\n    usable = []\n    for hkl, gamma in directed_gamma.items():\n        hkl = normalize_hkl(hkl)\n        gamma = float(gamma)\n        if hkl is None or not np.isfinite(gamma) or gamma <= 0:\n            continue\n        normal = direction_normal(lattice, hkl)\n        usable.append((hkl, normal, gamma))\n    if len(usable) < 4:\n        raise ValueError(f\"Only {len(usable)} positive oriented facets are available; at least four are needed.\")\n\n    # Collapse exact/near-identical normals only, preserving the lowest face distance.\n    unique = []\n    for hkl, normal, gamma in usable:\n        match = None\n        for i, (_, old_normal, _) in enumerate(unique):\n            if np.linalg.norm(normal - old_normal) < 1e-8:\n                match = i\n                break\n        if match is None:\n            unique.append((hkl, normal, gamma))\n        elif gamma < unique[match][2]:\n            unique[match] = (hkl, normal, gamma)\n    scale = min(item[2] for item in unique)\n    # scipy half-space convention: a.x + b <= 0, hence n.x - gamma/scale <= 0.\n    halfspaces = np.asarray([np.r_[normal, -gamma / scale] for _, normal, gamma in unique], dtype=float)\n    if not np.isfinite(halfspaces).all():\n        raise ValueError(\"Non-finite Wulff halfspaces.\")\n    try:\n        intersections = HalfspaceIntersection(halfspaces, np.zeros(3)).intersections\n        # Deduplicate near-coincident vertices for a stable hull.\n        vertices = []\n        for point in intersections:\n            if not any(np.linalg.norm(point - old) < 1e-6 for old in vertices):\n                vertices.append(np.asarray(point, dtype=float))\n        vertices = np.asarray(vertices, dtype=float)\n        if vertices.ndim != 2 or vertices.shape[1] != 3 or len(vertices) < 4:\n            raise ValueError(\"Half-space intersections did not produce a closed 3D polyhedron.\")\n        hull = ConvexHull(vertices)\n        if not np.isfinite(hull.volume) or hull.volume <= 0:\n            raise ValueError(\"Wulff polyhedron has no positive 3D volume.\")\n    except (QhullError, ValueError) as exc:\n        raise ValueError(\n            f\"Could not create a closed bounded polyhedron from these orientations: {type(exc).__name__}: {exc}\"\n        ) from exc\n    return {\n        \"label\": label,\n        \"vertices\": vertices,\n        \"simplices\": np.asarray(hull.simplices, dtype=int),\n        \"equations\": np.asarray(hull.equations, dtype=float),\n        \"volume_relative\": float(hull.volume),\n        \"facet_count_input\": int(len(unique)),\n        \"direction_gamma\": {tuple(x[0]): float(x[2]) for x in unique},\n    }\n\n\nmorphologies = []\nscenario_labels = {\n    \"minimum\": \"CHGNet_minimum_termination_Wulff\",\n    \"median\": \"CHGNet_median_termination_sensitivity_NOT_UNIQUE_EQUILIBRIUM\",\n    \"maximum\": \"CHGNet_maximum_termination_sensitivity_NOT_UNIQUE_EQUILIBRIUM\",\n}\nvalid_scenario_shapes = {}\nfor scenario in scenario_names:\n    gamma_map = direction_gamma_scenarios[scenario]\n    missing_reps = [hkl for hkl in miller_reps if hkl not in gamma_map]\n    if missing_reps:\n        if scenario == \"minimum\":\n            print(\"No complete CHGNet Wulff prediction: validated individual energies are missing for directed facet classes:\")\n            print(\", \".join(fmt_hkl(h) for h in missing_reps))\n            print(\"Missing energies are not replaced by arbitrary values. A partial-facet shape would be misleading.\")\n        continue\n    try:\n        _label = scenario_labels[scenario]\n        if scenario == \"minimum\" and estimated_directions:  # SYMMETRIC-SLAB CHANGE\n            _label = \"CHGNet_upper_bound_estimate_Wulff_PARTLY_PAIR_AVERAGED\"\n        item = build_polyhedron(_label, gamma_map, working_structure.lattice)\n        if scenario == \"minimum\":\n            morphologies.append(item)\n            valid_scenario_shapes[scenario] = item\n            print(\"Full-rank signed-orientation CHGNet Wulff polyhedron created.\")\n        if any(\n            (not fit_summary_df[(fit_summary_df[\"h\"] == rep[0]) & (fit_summary_df[\"k\"] == rep[1]) & (fit_summary_df[\"l\"] == rep[2])][\"has_redundant_equations\"].fillna(False).any())\n            for rep in miller_reps\n        ):\n            print(\"Note: at least one facet fit has no redundant pair equations; its algebraic solution is identifiable but lacks an internal residual-based consistency check.\")\n        else:\n            # Only keep sensitivity scenarios when some directional energy actually changes.\n            base = direction_gamma_scenarios[\"minimum\"]\n            changed = any(\n                not np.isclose(gamma_map.get(hkl, np.nan), base.get(hkl, np.nan), rtol=1e-5, atol=1e-8)\n                for hkl in miller_reps\n            )\n            if changed:\n                morphologies.append(item)\n                valid_scenario_shapes[scenario] = item\n                print(f\"Created {scenario} termination-sensitivity shape (not a unique equilibrium prediction).\")\n    except Exception as exc:\n        print(f\"{scenario} CHGNet Wulff construction failed:\", f\"{type(exc).__name__}: {exc}\")\n\nif not direction_gamma_J_m2:\n    print(\"No usable individual surface energies were identified; no CHGNet Wulff shape will be fabricated.\")\n\n# Geometry-only equal-gamma control, not a physical prediction.\nequal_gamma = {tuple(hkl): 1.0 for hkl in all_directed_hkls}\ntry:\n    morphologies.append(build_polyhedron(\"Equal_gamma_geometry_control_NOT_A_PREDICTION\", equal_gamma, working_structure.lattice))\nexcept Exception as exc:\n    print(\"Equal-gamma geometry control failed:\", f\"{type(exc).__name__}: {exc}\")\n\n# BFDH-like d_hkl geometry baseline; not a surface-energy calculation.\nbfdh_gamma = {}\nfor hkl in all_directed_hkls:\n    try:\n        d = float(working_structure.lattice.d_hkl(hkl))\n        if np.isfinite(d) and d > 0:\n            bfdh_gamma[tuple(hkl)] = d\n    except Exception:\n        continue\ntry:\n    morphologies.append(build_polyhedron(\"BFDH_like_d_hkl_geometry_baseline_NOT_MLIP\", bfdh_gamma, working_structure.lattice))\nexcept Exception as exc:\n    print(\"BFDH-like geometry baseline failed:\", f\"{type(exc).__name__}: {exc}\")\n\n# -----------------------------------------------------------------------------\n# 10. Render the real framework and atomistically clip it into each candidate shape.\n# -----------------------------------------------------------------------------\ntry:\n    import py3Dmol\nexcept Exception as exc:\n    py3Dmol = None\n    print(\"py3Dmol unavailable; files will still be exported:\", exc)\n\nbond_graph = None\nconnectivity_note = \"\"\ntry:\n    bond_graph = CrystalNN().get_bonded_structure(bulk_final)\n    connectivity_note = \"Bonds are inferred by CrystalNN for display; they are not guaranteed to be CIF-authored bond topology.\"\nexcept Exception as exc:\n    connectivity_note = f\"CrystalNN could not infer connectivity ({type(exc).__name__}: {exc}); atoms will be shown without bond cylinders.\"\n    print(\"Connectivity warning:\", connectivity_note)\n\n\ndef graph_edges_for_replicas(structure, graph, key_to_index, cell_ranges):\n    if graph is None:\n        return []\n    edges = set()\n    nsites = len(structure)\n    for cell in itertools.product(*cell_ranges):\n        cell = tuple(int(x) for x in cell)\n        for i in range(nsites):\n            skey = (i, cell)\n            if skey not in key_to_index:\n                continue\n            try:\n                connected_sites = graph.get_connected_sites(i)\n            except Exception:\n                continue\n            for connected in connected_sites:\n                j = int(connected.index)\n                image = tuple(int(v) for v in connected.jimage)\n                destination_cell = tuple(cell[q] + image[q] for q in range(3))\n                dkey = (j, destination_cell)\n                if dkey in key_to_index:\n                    a, b = key_to_index[skey], key_to_index[dkey]\n                    if a != b:\n                        edges.add((min(a, b), max(a, b)))\n    return sorted(edges)\n\n\ndef render_atoms(symbols, positions, edges=None, title=\"MOF\", hull_vertices=None, hull_simplices=None):\n    if py3Dmol is None or len(symbols) == 0:\n        return None\n    xyz_lines = [str(len(symbols)), title]\n    xyz_lines.extend(f\"{sym} {p[0]:.7f} {p[1]:.7f} {p[2]:.7f}\" for sym, p in zip(symbols, positions))\n    viewer = py3Dmol.view(width=900, height=650)\n    viewer.addModel(\"\\n\".join(xyz_lines), \"xyz\")\n    viewer.setStyle({}, {\"sphere\": {\"scale\": 0.27, \"colorscheme\": \"Jmol\"}})\n    if edges:\n        for i, j in edges:\n            p1, p2 = positions[i], positions[j]\n            try:\n                viewer.addCylinder({\n                    \"start\": {\"x\": float(p1[0]), \"y\": float(p1[1]), \"z\": float(p1[2])},\n                    \"end\": {\"x\": float(p2[0]), \"y\": float(p2[1]), \"z\": float(p2[2])},\n                    \"radius\": 0.065, \"color\": \"#888888\", \"fromCap\": 1, \"toCap\": 1,\n                })\n            except Exception as exc:\n                print(f\"Bond rendering stopped for {title}: {type(exc).__name__}: {exc}\")\n                break\n    if hull_vertices is not None and hull_simplices is not None:\n        hull_edges = set()\n        for face in hull_simplices:\n            for a, b in ((face[0], face[1]), (face[1], face[2]), (face[2], face[0])):\n                hull_edges.add((min(int(a), int(b)), max(int(a), int(b))))\n        for a, b in sorted(hull_edges):\n            p1, p2 = hull_vertices[a], hull_vertices[b]\n            try:\n                viewer.addCylinder({\n                    \"start\": {\"x\": float(p1[0]), \"y\": float(p1[1]), \"z\": float(p1[2])},\n                    \"end\": {\"x\": float(p2[0]), \"y\": float(p2[1]), \"z\": float(p2[2])},\n                    \"radius\": 0.04, \"color\": \"#222222\", \"fromCap\": 1, \"toCap\": 1,\n                })\n            except Exception as exc:\n                print(f\"Envelope rendering stopped for {title}: {type(exc).__name__}: {exc}\")\n                break\n    viewer.setBackgroundColor(\"white\")\n    viewer.zoomTo()\n    return viewer\n\n\ndef repeated_structure_view(structure, graph, repeats):\n    repeat_tuple = (repeats, repeats, repeats) if isinstance(repeats, int) else tuple(int(x) for x in repeats)\n    if len(repeat_tuple) != 3 or any(v < 1 for v in repeat_tuple):\n        raise ValueError(\"repeats must be a positive integer or positive 3-tuple\")\n    ranges = [range(repeat_tuple[0]), range(repeat_tuple[1]), range(repeat_tuple[2])]\n    matrix = np.asarray(structure.lattice.matrix, dtype=float)\n    symbols, positions, key_to_index = [], [], {}\n    for cell in itertools.product(*ranges):\n        cell = tuple(int(x) for x in cell)\n        shift = np.asarray(cell, dtype=float) @ matrix\n        for i, site in enumerate(structure):\n            key_to_index[(i, cell)] = len(positions)\n            symbols.append(site.specie.symbol)\n            positions.append(np.asarray(site.coords, dtype=float) + shift)\n    positions = np.asarray(positions, dtype=float)\n    positions -= positions.mean(axis=0)\n    edges = graph_edges_for_replicas(structure, graph, key_to_index, ranges)\n    return symbols, positions, edges\n\n\ndef make_atomistic_particle(morphology, structure, graph, radius_A):\n    raw_vertices = np.asarray(morphology[\"vertices\"], dtype=float)\n    if raw_vertices.ndim != 2 or raw_vertices.shape[1] != 3 or len(raw_vertices) < 4:\n        raise ValueError(\"Morphology has no valid 3D vertices\")\n    max_radius = float(np.max(np.linalg.norm(raw_vertices, axis=1)))\n    if not np.isfinite(max_radius) or max_radius <= 0:\n        raise ValueError(\"Invalid morphology scale\")\n    vertices = raw_vertices * (radius_A / max_radius)\n    hull = ConvexHull(vertices)\n    eq = hull.equations\n\n    matrix = np.asarray(structure.lattice.matrix, dtype=float)\n    inverse = np.linalg.inv(matrix)\n    center = np.mean(np.asarray(structure.cart_coords, dtype=float), axis=0)\n    base = np.asarray(structure.cart_coords, dtype=float) - center\n    base_frac = base @ inverse\n    vertex_frac = vertices @ inverse\n    extent = np.ceil(np.max(np.abs(vertex_frac), axis=0) + np.max(np.abs(base_frac), axis=0)).astype(int) + 1\n    ranges = [range(-int(n), int(n) + 1) for n in extent]\n\n    symbols, xyz, key_to_index = [], [], {}\n    for cell in itertools.product(*ranges):\n        cell = tuple(int(x) for x in cell)\n        shift = np.asarray(cell, dtype=float) @ matrix\n        for i, site in enumerate(structure):\n            pos = base[i] + shift\n            if np.max(eq[:, :3] @ pos + eq[:, 3]) <= 1e-7:\n                key = (i, cell)\n                key_to_index[key] = len(xyz)\n                symbols.append(site.specie.symbol)\n                xyz.append(pos)\n    if not xyz:\n        raise ValueError(\"No atoms lie inside this polyhedron; increase the visualization radius or inspect the lattice setting\")\n    xyz = np.asarray(xyz, dtype=float)\n\n    # Remove duplicate coordinates with a deterministic atom-index remapping.\n    keep_indices = []\n    seen = set()\n    for i, (symbol, pos) in enumerate(zip(symbols, xyz)):\n        key = (symbol, *np.round(pos, 6))\n        if key not in seen:\n            seen.add(key)\n            keep_indices.append(i)\n    if len(keep_indices) != len(xyz):\n        old_to_new = {old: new for new, old in enumerate(keep_indices)}\n        xyz = xyz[keep_indices]\n        symbols = [symbols[i] for i in keep_indices]\n        key_to_index = {key: old_to_new[idx] for key, idx in key_to_index.items() if idx in old_to_new}\n\n    mins, maxs = xyz.min(axis=0), xyz.max(axis=0)\n    lengths = np.maximum(maxs - mins + 10.0, 10.0)\n    frac = (xyz - mins + 5.0) / lengths\n    particle = Structure(\n        Lattice.from_parameters(float(lengths[0]), float(lengths[1]), float(lengths[2]), 90, 90, 90),\n        symbols, frac, coords_are_cartesian=False,\n    )\n    stem = re.sub(r\"[^A-Za-z0-9_.-]+\", \"_\", morphology[\"label\"])\n    cif_path = PARTICLE_DIR / f\"{safe_name}_{stem}_atomistic_particle.cif\"\n    xyz_path = PARTICLE_DIR / f\"{safe_name}_{stem}_atomistic_particle.xyz\"\n    particle.to(filename=str(cif_path))\n    Molecule(symbols, xyz).to(filename=str(xyz_path))\n    edges = graph_edges_for_replicas(structure, graph, key_to_index, ranges)\n    return {\n        \"label\": morphology[\"label\"], \"symbols\": symbols, \"xyz\": xyz, \"edges\": edges,\n        \"vertices\": vertices, \"simplices\": hull.simplices, \"cif_path\": str(cif_path),\n        \"xyz_path\": str(xyz_path), \"atom_count\": len(symbols),\n    }\n\nmorphology_rows = []\nparticle_objects = []\nfor morphology in morphologies:\n    try:\n        particle = make_atomistic_particle(morphology, bulk_final, bond_graph, PARTICLE_RADIUS_A)\n        particle_objects.append(particle)\n        methodology = (\n            \"CHGNet Wulff using minimum fitted termination gamma per signed orientation\"\n            if morphology[\"label\"].startswith(\"CHGNet_minimum\") else\n            \"CHGNet Wulff from upper-bound estimates (partial fits and/or half the lowest pair excess); not fully fitted\"\n            if morphology[\"label\"].startswith(\"CHGNet_upper_bound\") else\n            \"Termination-energy sensitivity scenario; not a unique equilibrium prediction\"\n            if morphology[\"label\"].startswith(\"CHGNet\") else\n            \"BFDH-like d_hkl geometry baseline; not a surface-energy calculation\"\n            if morphology[\"label\"].startswith(\"BFDH\") else\n            \"Equal-gamma geometry control; not a physical morphology prediction\"\n        )\n        morphology_rows.append({\n            \"morphology\": morphology[\"label\"], \"method\": methodology,\n            \"atom_count\": particle[\"atom_count\"], \"display_radius_A\": PARTICLE_RADIUS_A,\n            \"relative_polyhedron_volume\": morphology[\"volume_relative\"],\n            \"cif_file\": particle[\"cif_path\"], \"xyz_file\": particle[\"xyz_path\"],\n            \"status\": \"ok\", \"detail\": \"Actual CIF-derived atoms clipped geometrically; not passivated or nanoparticle-relaxed.\"\n        })\n        print(f\"Created {morphology['label']}: {particle['atom_count']} atoms\")\n    except Exception as exc:\n        morphology_rows.append({\n            \"morphology\": morphology.get(\"label\", \"unknown\"), \"method\": \"atomistic visualization\",\n            \"atom_count\": 0, \"display_radius_A\": PARTICLE_RADIUS_A,\n            \"relative_polyhedron_volume\": morphology.get(\"volume_relative\", np.nan),\n            \"cif_file\": \"\", \"xyz_file\": \"\", \"status\": \"particle_construction_failed\",\n            \"detail\": f\"{type(exc).__name__}: {exc}\"\n        })\n        print(f\"Particle construction failed for {morphology.get('label')}: {type(exc).__name__}: {exc}\")\n\nmorphology_df = pd.DataFrame(morphology_rows)\nmorphology_csv = RESULTS_DIR / f\"{safe_name}_morphology_comparison.csv\"\nmorphology_df.to_csv(morphology_csv, index=False)\n\n# -----------------------------------------------------------------------------\n# 11. Interactive outputs: actual network, representative slabs, and morphology gallery.\n# -----------------------------------------------------------------------------\nprint(\"\\n================ PERIODIC MOF NETWORK ================\")\nprint(connectivity_note)\nbulk_symbols, bulk_positions, bulk_edges = repeated_structure_view(bulk_final, bond_graph, DISPLAY_REPEATS)\nbulk_viewer = render_atoms(\n    bulk_symbols, bulk_positions, bulk_edges,\n    title=f\"{formula} — {DISPLAY_REPEATS}x{DISPLAY_REPEATS}x{DISPLAY_REPEATS} periodic MOF network\",\n)\nif bulk_viewer is not None:\n    display(bulk_viewer)\n\nprint(\"\\n================ REPRESENTATIVE RELAXED SURFACE SLABS ================\")\nshown = set()\nfor row in records_df.itertuples():\n    if not getattr(row, \"relaxed_slab_file\", \"\") or not isinstance(row.relaxed_slab_file, str):\n        continue\n    facet_key = (int(row.h), int(row.k), int(row.l))\n    if facet_key in shown or not os.path.isfile(row.relaxed_slab_file):\n        continue\n    shown.add(facet_key)\n    try:\n        slab_struct = Structure.from_file(row.relaxed_slab_file)\n        slab_graph = None\n        try:\n            slab_graph = CrystalNN().get_bonded_structure(slab_struct)\n        except Exception:\n            pass\n        # Display a small lateral repeat only; slab CIF retains the full generated cell.\n        syms, pos, edges = repeated_structure_view(slab_struct, slab_graph, (2, 2, 1))\n        print(f\"Facet class {row.miller}; termination {row.termination}; status {row.status}\")\n        viewer = render_atoms(syms, pos, edges, title=f\"MOF slab {row.miller}, termination {row.termination}\")\n        if viewer is not None:\n            display(viewer)\n    except Exception as exc:\n        print(f\"Slab visualization failed for {getattr(row, 'miller', '?')}: {type(exc).__name__}: {exc}\")\n\nprint(\"\\n================ ATOMISTIC MORPHOLOGY GALLERY ================\")\nfor particle in particle_objects:\n    print(f\"{particle['label']} — {particle['atom_count']} atoms\")\n    viewer = render_atoms(\n        particle[\"symbols\"], particle[\"xyz\"], particle[\"edges\"],\n        title=particle[\"label\"], hull_vertices=particle[\"vertices\"], hull_simplices=particle[\"simplices\"],\n    )\n    if viewer is not None:\n        display(viewer)\nif not particle_objects:\n    print(\"No atomistic morphology particles were created. Check polyhedron and clipping diagnostics.\")\n\n# -----------------------------------------------------------------------------\n# 12. Metadata, summary, and ZIP export.\n# -----------------------------------------------------------------------------\nmetadata = {\n    \"project\": \"MorphoGraph-MOF\",\n    \"input_cif\": os.path.abspath(CIF_PATH),\n    \"formula\": formula,\n    \"input_atom_count\": int(len(input_structure)),\n    \"working_cell_atom_count\": int(len(working_structure)),\n    \"working_lattice_matrix_A\": np.asarray(working_structure.lattice.matrix).tolist(),\n    \"standardization_notes\": standardization_notes,\n    \"parameters\": {\n        \"max_miller_index\": MAX_MILLER_INDEX,\n        \"slab_thickness_A\": SLAB_THICKNESS_A,\n        \"vacuum_A\": VACUUM_A,\n        \"force_tolerance_eV_A\": FORCE_TOL_EV_A,\n        \"max_relax_steps\": MAX_RELAX_STEPS,\n        \"particle_display_radius_A_visualization_only\": PARTICLE_RADIUS_A,\n        \"bulk_view_repeats_per_axis\": DISPLAY_REPEATS,\n        \"symmetry_tolerance_A\": SYMPREC_A,\n        \"surface_fingerprint_depth_A\": SURFACE_FINGERPRINT_DEPTH_A,\n        \"fingerprint_distance_quantization_A\": FINGERPRINT_DISTANCE_TOL_A,\n        \"pair_fit_rms_tolerance_eV_A2\": PAIR_FIT_RMS_TOL_EV_A2,\n        \"termination_shift_deduplication_tolerance_fractional\": TERMINATION_SHIFT_TOL_FRACTIONAL,\n    },\n    \"bulk_reference\": {\n        \"available\": bool(model is not None), \"converged\": bool(bulk_ok),\n        \"energy_total_eV\": bulk_total_energy_eV, \"energy_per_atom_eV\": bulk_energy_per_atom_eV,\n        \"max_force_eV_A\": bulk_fmax_eV_A, \"error\": bulk_error,\n    },\n    \"miller_representatives\": [list(x) for x in miller_reps],\n    \"directed_miller_orbits\": {str(k): [list(x) for x in v] for k, v in miller_orbits.items()},\n    \"miller_generation_notes\": miller_notes,\n    \"surface_pair_record_count\": int(len(records_df)),\n    \"surface_pair_status_counts\": {str(k): int(v) for k, v in records_df[\"status\"].value_counts().items()} if len(records_df) else {},\n    \"surface_fit_summary\": fit_summary_df.to_dict(orient=\"records\"),\n    \"individual_surface_energy_count\": int(len(surface_gamma_df)),\n    \"valid_signed_direction_energy_count\": int(len(direction_gamma_J_m2)),\n    \"direction_energy_source\": {fmt_hkl(d): s for d, s in direction_gamma_source.items()},  # SYMMETRIC-SLAB CHANGE\n    \"direction_gamma_J_m2\": {fmt_hkl(d): g for d, g in direction_gamma_J_m2.items()},\n    \"direction_energy_notes\": direction_gamma_notes,\n    \"morphologies\": morphology_rows,\n    \"connectivity_note\": connectivity_note,\n    \"methodology\": [\n        \"Stoichiometric slabs only are used for the simple bulk-reference pair-energy equations.\",\n        \"A relaxed slab verified symmetric by pymatgen is constrained to one shared top/bottom surface-state unknown in the pair-energy equations.\",\n        \"For asymmetric slabs, (E_slab - N*E_bulk)/A is treated as gamma_top + gamma_bottom, not as one surface energy.\",\n        \"Slab generation tries the primitive surface cell first and retries with a non-primitive surface cell if slab generation raises an exception; the chosen mode is recorded per facet.\",\n        \"Individual termination energies are returned only when the surface-pair linear system, fingerprinted from relaxed surface states, is full rank; if redundant equations exist, their RMS residual must meet the explicit user tolerance.\",\n        \"A full-rank square pair system can identify energies algebraically but has no independent redundancy-based residual validation; this is marked in the fit CSV and requires external convergence/validation.\",\n        \"A CHGNet Wulff shape is created only when every requested directed facet class has valid positive surface-energy estimates.\",\n        \"SYMMETRIC-SLAB VERSION: symmetrized slabs are generated in addition to the as-cut slabs; non-stoichiometric ones are still excluded. A slab symmetric before relaxation shares one unknown for both faces.\",\n        \"SYMMETRIC-SLAB VERSION: termination energies fixed by the null-space test are kept from underdetermined systems; facet classes with none fall back to half their lowest pair excess. Both are upper bounds on the facet's lowest termination energy, recorded in direction_energy_source.\",\n        \"No missing MLIP surface energies are filled with equal, mirrored, random, or guessed values.\",\n        \"BFDH-like and equal-gamma shapes are geometry controls, not MLIP surface-energy predictions.\",\n        \"Atomistic particles are geometric cuts of the periodic bulk; they are not passivated or separately relaxed nanoparticles.\",\n        \"CrystalNN connectivity is inferred for display and must be chemically checked for unusual metal coordination or disordered structures.\",\n        \"CHGNet is not automatically validated for a given MOF; validate representative surface energies against DFT-D3 and perform slab-thickness/vacuum convergence studies.\",\n        \"This code creates training labels/baselines; it does not train the proposed cross-MOF GNN.\",\n    ],\n}\nmetadata_path = RESULTS_DIR / f\"{safe_name}_metadata.json\"\nwith open(metadata_path, \"w\", encoding=\"utf-8\") as f:\n    json.dump(metadata, f, indent=2, default=str)\n\nprint(\"\\n================ FINAL SUMMARY ================\")\nprint(f\"Bulk reference converged: {bulk_ok}\")\nprint(f\"Pair-energy records     : {len(records_df)}\")\nprint(f\"Individual termination gammas: {len(surface_gamma_df)}\")\nprint(f\"Signed direction gammas : {len(direction_gamma_J_m2)} / {len(all_directed_hkls)} directed primitive indices\")\nprint(\"Surface-pair statuses:\")\nif len(records_df):\n    print(records_df[\"status\"].value_counts(dropna=False).to_string())\nprint(\"Morphology outputs:\")\nif len(morphology_df):\n    display(morphology_df)\nprint(\"\\nOutput files:\")\nfor path in (records_csv, fit_csv, surface_gamma_csv, morphology_csv, metadata_path):\n    print(path.resolve())\n\narchive_path = Path(f\"{safe_name}_MorphoGraph_MOF_revised_results_{RUN_ID}.zip\")\nwith ZipFile(archive_path, \"w\", compression=ZIP_DEFLATED) as zf:\n    for path in RESULTS_DIR.rglob(\"*\"):\n        if path.is_file():\n            zf.write(path, arcname=str(Path(RESULTS_DIR.name) / path.relative_to(RESULTS_DIR)))\nprint(f\"\\nZIP archive created: {archive_path.resolve()}\")\ntry:\n    from google.colab import files\n    pass  # download handled by the driver cell\nexcept Exception:\n    print(\"Outside Colab: retrieve the ZIP from the working directory.\")\n"


In [ ]:
# ---- DRIVER: run each MOF in its own namespace and keep going if one fails -----------
import time, glob, traceback
summary = []
for mof in MOFS_TO_RUN:
    print("\n" + "=" * 90 + f"\n{mof}\n" + "=" * 90)
    t0 = time.time()
    ns = {"__name__": "__main__", "CIF_PATH": f"{mof}.cif", "SETTINGS": dict(SETTINGS)}
    status = "finished"
    try:
        exec(compile(MAIN_SRC, f"<MorphoGraph {mof}>", "exec"), ns)
    except Exception:
        status = "FAILED"
        traceback.print_exc()
    minutes = (time.time() - t0) / 60
    zips = sorted(glob.glob(f"{mof}_MorphoGraph_MOF_revised_results_*.zip"))
    summary.append({"MOF": mof, "status": status, "minutes": round(minutes, 1),
                    "bulk_converged": ns.get("bulk_ok"), "zip": zips[-1] if zips else None})
    print(f"\n{mof}: {status} in {minutes:.1f} min")
    if zips:
        try:
            from google.colab import files
            files.download(zips[-1])
        except Exception:
            print("Not on Colab: zip left in", zips[-1])

import pandas as pd
display(pd.DataFrame(summary))
